# Notebook 02 — Final RibFrac 500-Patient Preprocessing and Patient Registry

This notebook is the **final preprocessing/registry stage** for the RibFrac labelled development cohort.

## Source cohort
- Training Part 1: 300 CTs
- Training Part 2: 120 CTs
- Tuning/Validation: 80 CTs
- Total labelled patients: **500**

## Final patient-level protocol
The three labelled sources are combined first and then split at the **patient level**:
- Training: 350 patients
- Validation: 75 patients
- Internal test: 75 patients

The internal test set is distinct from the official RibFrac challenge test set, whose ground truth is not publicly available.

## Important storage design
The full 1-mm isotropic preprocessed volumes are **not permanently materialized for all 500 patients** in `/kaggle/working`. Instead, this notebook:
1. validates all source metadata;
2. creates and saves the final patient registry/split;
3. provides a reproducible, on-demand patient preprocessing function;
4. optionally materializes selected patients for smoke testing.

Notebook 03 can use the same on-demand preprocessing logic to extract patches patient-by-patient, avoiding a 100+ GB intermediate volume store.


In [1]:
#%pip install -q kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 1 — Imports
import gc
import hashlib
import io
import json
import zlib
import tempfile
import re
import os
import random
import struct
import time
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import kagglehub

print("Libraries imported successfully.")


Libraries imported successfully.


In [3]:

# ============================================================
# NOTEBOOK 02 — EXISTING DATASET AND SPLIT AUDIT
# ============================================================

from pathlib import Path
import pandas as pd

DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

SAVED_SPLIT = (
    DATASET_ROOT / "metadata" / "patient_split_500.csv"
)

if not SAVED_SPLIT.is_file():
    raise FileNotFoundError(
        "Existing patient split not found. "
        "Attach the persisted RibFrac dataset in Kaggle "
        "before proceeding."
    )

# ------------------------------------------------------------
# 1. Load saved patient assignments
# ------------------------------------------------------------

saved_split = pd.read_csv(SAVED_SPLIT)

id_col = (
    "public_id"
    if "public_id" in saved_split.columns
    else "patient_id"
)

saved_split = saved_split.rename(
    columns={id_col: "patient_id"}
)

assert "split" in saved_split.columns

saved_split["patient_id"] = (
    saved_split["patient_id"].astype(str)
)

saved_split["split"] = (
    saved_split["split"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"val": "validation"})
)

assert len(saved_split) == 500
assert saved_split["patient_id"].nunique() == 500

expected = {
    "train": 350,
    "validation": 75,
    "test": 75
}

assert saved_split["split"].value_counts().to_dict() == expected

print("=" * 60)
print("EXISTING PATIENT SPLIT")
print("=" * 60)

print(saved_split["split"].value_counts())

# ------------------------------------------------------------
# 2. Check existing preprocessed arrays
# ------------------------------------------------------------

image_dir = DATASET_ROOT / "images"
label_dir = DATASET_ROOT / "labels"

image_ids = {
    path.name.removesuffix("-image.npy")
    for path in image_dir.glob("*-image.npy")
}

label_ids = {
    path.name.removesuffix("-label.npy")
    for path in label_dir.glob("*-label.npy")
}

complete_ids = image_ids & label_ids

print("\nPersisted arrays:")
print("CT files:", len(image_ids))
print("Label files:", len(label_ids))
print("Complete patient pairs:", len(complete_ids))

print("CT files without labels:", len(image_ids - label_ids))
print("Labels without CT:", len(label_ids - image_ids))

# ------------------------------------------------------------
# 3. Check which splits the saved arrays belong to
# ------------------------------------------------------------

existing_patients = saved_split[
    saved_split["patient_id"].isin(complete_ids)
]

assert len(existing_patients) == len(complete_ids), (
    "Some saved patients are absent from the split."
)

print("\nSaved patients by split:")
print(existing_patients["split"].value_counts())

print("\nExisting complete patient IDs:")
print(sorted(complete_ids))

print("\nDATASET AUDIT COMPLETED")
print("No files or assignments were modified.")


EXISTING PATIENT SPLIT
split
train         350
validation     75
test           75
Name: count, dtype: int64

Persisted arrays:
CT files: 13
Label files: 13
Complete patient pairs: 13
CT files without labels: 0
Labels without CT: 0

Saved patients by split:
split
test    13
Name: count, dtype: int64

Existing complete patient IDs:
['RibFrac107', 'RibFrac109', 'RibFrac114', 'RibFrac12', 'RibFrac120', 'RibFrac121', 'RibFrac129', 'RibFrac134', 'RibFrac140', 'RibFrac149', 'RibFrac15', 'RibFrac156', 'RibFrac167']

DATASET AUDIT COMPLETED
No files or assignments were modified.


In [5]:

# ============================================================
# NOTEBOOK 02 — VERIFY SAVED PATIENT ASSIGNMENTS
# ============================================================


# Existing published Kaggle dataset
PERSISTED_ROOT = Path(
    "/kaggle/input/datasets/waboke/"
    "ribfrac-preprocessed-final"
)

REFERENCE_SPLIT = (
    PERSISTED_ROOT / "metadata" / "patient_split_500.csv"
)

# Split generated in Notebook 01, if accessible
NOTEBOOK01_SPLIT = Path(
    "/kaggle/working/ribfrac_metadata/"
    "patient_split_500.csv"
)

# Current working split from Notebook 02
NOTEBOOK02_SPLIT = Path(
    "/kaggle/working/ribfrac_preprocessed_final/"
    "metadata/patient_split_500.csv"
)


def load_split(path):

    df = pd.read_csv(path)

    if "public_id" in df.columns:
        df = df.rename(
            columns={"public_id": "patient_id"}
        )

    df = df[["patient_id", "split"]].copy()

    df["patient_id"] = df["patient_id"].astype(str)

    df["split"] = (
        df["split"].astype(str)
        .str.strip().str.lower()
        .replace({"val": "validation"})
    )

    assert len(df) == 500
    assert df["patient_id"].nunique() == 500

    return df.set_index("patient_id")["split"].sort_index()


reference = load_split(REFERENCE_SPLIT)

print("=" * 60)
print("REFERENCE: PERSISTED DATASET SPLIT")
print("=" * 60)

print(reference.value_counts())


def compare_split(name, path):

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    if not path.is_file():

        print("File not available in this Kaggle session.")
        print("No comparison performed.")
        return

    candidate = load_split(path)

    if not candidate.index.equals(reference.index):
        print("PATIENT IDs DO NOT MATCH")
        return

    mismatch = reference != candidate

    count = int(mismatch.sum())

    print("Different assignments:", count)

    if count == 0:

        print("SPLIT CONSISTENCY: PASS")

    else:

        print("SPLIT CONSISTENCY: MISMATCH")

        differences = pd.DataFrame({
            "persisted_split": reference[mismatch],
            "other_split": candidate[mismatch]
        })

        display(differences.head(20))

        print(
            "Do not overwrite the persisted split."
        )


compare_split(
    "NOTEBOOK 01 SPLIT",
    NOTEBOOK01_SPLIT
)

compare_split(
    "CURRENT NOTEBOOK 02 SPLIT",
    NOTEBOOK02_SPLIT
)

print("\nSplit comparison completed.")
print("No files were modified.")


REFERENCE: PERSISTED DATASET SPLIT
split
train         350
validation     75
test           75
Name: count, dtype: int64

NOTEBOOK 01 SPLIT
File not available in this Kaggle session.
No comparison performed.

CURRENT NOTEBOOK 02 SPLIT
File not available in this Kaggle session.
No comparison performed.

Split comparison completed.
No files were modified.


In [6]:
# Cell 2 — Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Random seed:", SEED)


Random seed: 42


In [7]:
# Cell 3 — Final source configuration

# Official Zenodo records
SOURCES = {
    "train_part1": {
        "record": "3893508",
        "expected_patients": 300,
        "info_filename": "ribfrac-train-info-1.csv",
        "image_zip": "ribfrac-train-images-1.zip",
        "label_zip": "ribfrac-train-labels-1.zip",
        "image_url": "https://zenodo.org/records/3893508/files/ribfrac-train-images-1.zip",
        "label_url": "https://zenodo.org/records/3893508/files/ribfrac-train-labels-1.zip",
        "info_url": "https://zenodo.org/records/3893508/files/ribfrac-train-info-1.csv",
    },
    "train_part2": {
        "record": "3893498",
        "expected_patients": 120,
        "info_filename": "ribfrac-train-info-2.csv",
        "image_zip": "ribfrac-train-images-2.zip",
        "label_zip": "ribfrac-train-labels-2.zip",
        "image_url": "https://zenodo.org/records/3893498/files/ribfrac-train-images-2.zip",
        "label_url": "https://zenodo.org/records/3893498/files/ribfrac-train-labels-2.zip",
        "info_url": "https://zenodo.org/records/3893498/files/ribfrac-train-info-2.csv",
    },
    "validation": {
        "record": "3893496",
        "expected_patients": 80,
        "info_filename": "ribfrac-val-info.csv",
        "image_zip": "ribfrac-val-images.zip",
        "label_zip": "ribfrac-val-labels.zip",
        "image_url": "https://zenodo.org/records/3893496/files/ribfrac-val-images.zip",
        "label_url": "https://zenodo.org/records/3893496/files/ribfrac-val-labels.zip",
        "info_url": "https://zenodo.org/records/3893496/files/ribfrac-val-info.csv",
    },
}

TOTAL_EXPECTED = sum(v["expected_patients"] for v in SOURCES.values())

print("Expected labelled patients:", TOTAL_EXPECTED)
for name, cfg in SOURCES.items():
    print(f"{name}: {cfg['expected_patients']} patients")


Expected labelled patients: 500
train_part1: 300 patients
train_part2: 120 patients
validation: 80 patients


In [73]:
# Cell 4 — Persistent working/output directories

WORK_ROOT = Path("/kaggle/working/ribfrac_preprocessed_final")
IMAGE_ROOT = WORK_ROOT / "images"
LABEL_ROOT = WORK_ROOT / "labels"
METADATA_ROOT = WORK_ROOT / "metadata"
CACHE_ROOT = WORK_ROOT / "cache"
SMOKE_ROOT = WORK_ROOT / "smoke_test"

for path in [WORK_ROOT, IMAGE_ROOT, LABEL_ROOT, METADATA_ROOT, CACHE_ROOT, SMOKE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# Process one or a small number of patients per run while validating the pipeline.
# Set to None only when the available persistent storage is sufficient.
MAX_PATIENTS_PER_RUN = 5
RESUME_EXISTING = True
RETRY_FAILED = True

print("Persistent root:", WORK_ROOT)
print("Images:", IMAGE_ROOT)
print("Labels:", LABEL_ROOT)
print("Metadata:", METADATA_ROOT)
print("MAX_PATIENTS_PER_RUN:", MAX_PATIENTS_PER_RUN)
print("RESUME_EXISTING:", RESUME_EXISTING)


Persistent root: /kaggle/working/ribfrac_preprocessed_final
Images: /kaggle/working/ribfrac_preprocessed_final/images
Labels: /kaggle/working/ribfrac_preprocessed_final/labels
Metadata: /kaggle/working/ribfrac_preprocessed_final/metadata
MAX_PATIENTS_PER_RUN: 5
RESUME_EXISTING: True


## Source access strategy

Zenodo's image archives are large (for example, Part 1 is 36.6 GB and Part 2 is 14.6 GB), so the notebook does **not** download an entire image archive. citeturn0view0turn1search0

Instead, image archives are accessed using HTTP byte ranges and only the requested patient's ZIP member is downloaded. The small annotation archives are cached locally.


In [74]:

# Cell 5 — HTTP helpers

def http_get_bytes(url, start=None, end=None, timeout=180, max_retries=6):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Encoding": "identity",
    }

    if start is not None and end is not None:
        headers["Range"] = f"bytes={start}-{end}"

    last_error = None

    for attempt in range(max_retries):
        try:
            request = Request(url, headers=headers)

            with urlopen(request, timeout=timeout) as response:
                data = response.read()
                response_headers = response.headers
                status = response.status

            # A range request must return partial content.
            if start is not None and end is not None:
                if status != 206:
                    content_range = response_headers.get("Content-Range")
                    if not (
                        status == 200
                        and start == 0
                        and len(data) == end - start + 1
                    ):
                        raise RuntimeError(
                            f"Expected HTTP 206 for byte range "
                            f"{start}-{end}, got HTTP {status}; "
                            f"Content-Range={content_range}"
                        )

                expected_length = end - start + 1
                if len(data) != expected_length:
                    raise RuntimeError(
                        f"Incomplete range response: received {len(data)} "
                        f"bytes, expected {expected_length}"
                    )

            return data, response_headers

        except HTTPError as exc:
            last_error = exc

            # Retry temporary server errors and rate limiting.
            if exc.code not in (408, 429, 500, 502, 503, 504):
                raise

        except (URLError, TimeoutError, ConnectionError) as exc:
            last_error = exc

        if attempt < max_retries - 1:
            wait_seconds = min(60, 2 ** attempt) + random.uniform(0, 1)
            print(
                f"Download attempt {attempt + 1}/{max_retries} failed: "
                f"{last_error}. Retrying in {wait_seconds:.1f}s..."
            )
            time.sleep(wait_seconds)

    raise RuntimeError(
        f"Download failed after {max_retries} attempts. "
        f"Last error: {last_error}"
    ) from last_error
print("HTTP helpers defined.")

HTTP helpers defined.


In [75]:

# ============================================================
# CELL #5A — FIX MISSING get_remote_size() FUNCTION
# ============================================================

from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import re

def get_remote_size(url, timeout=120):
    """
    Retrieve remote ZIP file size without downloading
    the entire archive.
    """

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Encoding": "identity"
    }

    # First attempt: HEAD request
    try:
        request = Request(
            url,
            headers=headers,
            method="HEAD"
        )

        with urlopen(request, timeout=timeout) as response:
            size = response.headers.get("Content-Length")

            if size is not None and int(size) > 0:
                return int(size)

    except (HTTPError, URLError, OSError, ValueError) as e:
        print("HEAD request unavailable:", e)

    # Second attempt: HTTP byte-range request
    headers["Range"] = "bytes=0-0"

    request = Request(url, headers=headers)

    with urlopen(request, timeout=timeout) as response:
        if response.status != 206:
            raise RuntimeError(
                "Server did not honor the byte-range request. "
                f"HTTP status: {response.status}"
            )

        content_range = response.headers.get(
            "Content-Range", ""
        )

        match = re.search(r"/(\d+)$", content_range)

        if match is None:
            raise RuntimeError(
                f"Invalid Content-Range: {content_range}"
            )

        return int(match.group(1))


print("get_remote_size() defined successfully.")


get_remote_size() defined successfully.


In [76]:
print("Remote size helper:", callable(get_remote_size))
print("HTTP reader:", callable(http_get_bytes))
print("ZIP reader:", callable(read_zip64_central_directory))
print("Patient loader:", callable(load_patient_nifti_from_source))

Remote size helper: True
HTTP reader: True
ZIP reader: True
Patient loader: True


In [77]:
# Cell 6 — ZIP central-directory inspection

def parse_zip64_extra(extra):
    # ZIP64 extra field ID = 0x0001
    pos = 0
    values = []
    while pos + 4 <= len(extra):
        header_id, size = struct.unpack("<HH", extra[pos:pos+4])
        data = extra[pos+4:pos+4+size]
        pos += 4 + size
        if header_id == 0x0001:
            values.append(data)
    return values


def read_zip64_central_directory(url, chunk_size=1024 * 1024):
    size = get_remote_size(url)

    tail_size = min(size, 1024 * 1024)
    tail, _ = http_get_bytes(url, size - tail_size, size - 1)

    # EOCD signature
    eocd_sig = b"PK\x05\x06"
    eocd_pos = tail.rfind(eocd_sig)

    if eocd_pos < 0:
        raise RuntimeError("ZIP End Of Central Directory not found.")

    eocd = tail[eocd_pos:eocd_pos + 22]
    _, disk, cd_disk, entries_disk, entries_total, cd_size32, cd_offset32, comment_len = struct.unpack(
        "<4sHHHHIIH", eocd
    )

    if cd_size32 != 0xFFFFFFFF and cd_offset32 != 0xFFFFFFFF:
        cd_size = cd_size32
        cd_offset = cd_offset32
        entries_total = entries_total
    else:
        # ZIP64 EOCD locator
        locator_sig = b"PK\x06\x07"
        locator_pos = tail.rfind(locator_sig, 0, eocd_pos)
        if locator_pos < 0:
            raise RuntimeError("ZIP64 EOCD locator not found.")

        _, _, zip64_eocd_offset, _ = struct.unpack(
            "<4sIQI", tail[locator_pos:locator_pos + 20]
        )

        read_size = 1024 * 1024
        z64, _ = http_get_bytes(
            url,
            zip64_eocd_offset,
            min(size - 1, zip64_eocd_offset + read_size - 1),
        )

        if z64[:4] != b"PK\x06\x06":
            raise RuntimeError("ZIP64 EOCD record not found.")

        entries_total = struct.unpack_from("<Q", z64, 32)[0]
        cd_size = struct.unpack_from("<Q", z64, 40)[0]
        cd_offset = struct.unpack_from("<Q", z64, 48)[0]

    cd_bytes, _ = http_get_bytes(
        url,
        cd_offset,
        cd_offset + cd_size - 1,
    )

    entries = {}
    pos = 0

    for _ in range(entries_total):
        if cd_bytes[pos:pos+4] != b"PK\x01\x02":
            raise RuntimeError("Invalid central-directory signature.")

        fixed = cd_bytes[pos:pos+46]
        fields = struct.unpack("<4s6H3I5H2I", fixed)

        (
            signature,
            version_made,
            version_needed,
            flags,
            compression,
            mod_time,
            mod_date,
            crc32,
            comp_size32,
            uncomp_size32,
            name_len,
            extra_len,
            comment_len,
            disk_start,
            int_attr,
            ext_attr,
            local_offset32,
        ) = fields

        name_start = pos + 46
        name = cd_bytes[name_start:name_start + name_len].decode(
            "utf-8", errors="replace"
        )

        extra_start = name_start + name_len
        extra = cd_bytes[extra_start:extra_start + extra_len]

        comp_size = comp_size32
        uncomp_size = uncomp_size32
        local_offset = local_offset32

        if (
            comp_size32 == 0xFFFFFFFF
            or uncomp_size32 == 0xFFFFFFFF
            or local_offset32 == 0xFFFFFFFF
        ):
            zip64_chunks = parse_zip64_extra(extra)
            if not zip64_chunks:
                raise RuntimeError(f"ZIP64 extra field missing for {name}")
            data = zip64_chunks[0]
            zpos = 0
            if uncomp_size32 == 0xFFFFFFFF:
                uncomp_size = struct.unpack_from("<Q", data, zpos)[0]
                zpos += 8
            if comp_size32 == 0xFFFFFFFF:
                comp_size = struct.unpack_from("<Q", data, zpos)[0]
                zpos += 8
            if local_offset32 == 0xFFFFFFFF:
                local_offset = struct.unpack_from("<Q", data, zpos)[0]
                zpos += 8

        entries[name] = {
            "compression": compression,
            "compressed_size": int(comp_size),
            "uncompressed_size": int(uncomp_size),
            "local_header_offset": int(local_offset),
        }

        pos = extra_start + extra_len + comment_len

    return size, entries


print("Remote ZIP inspection helper defined.")


Remote ZIP inspection helper defined.


In [78]:
# Cell 7 — Patient member discovery

def normalize_member_name(name):
    return name.lstrip("/").replace("\\", "/")


def find_patient_member(entries, patient_id, kind):
    patient_id = str(patient_id)

    candidates = []
    for name in entries:
        base = Path(normalize_member_name(name)).name
        if patient_id in base:
            lower = base.lower()
            if kind == "image" and "image" in lower:
                candidates.append(name)
            elif kind == "label" and "label" in lower:
                candidates.append(name)

    if not candidates:
        raise FileNotFoundError(
            f"No {kind} ZIP member found for patient {patient_id}."
        )

    if len(candidates) > 1:
        # Prefer exact conventional filename.
        exact = [
            n for n in candidates
            if Path(n).name == f"{patient_id}-image.nii.gz"
            or Path(n).name == f"{patient_id}-label.nii.gz"
        ]
        if len(exact) == 1:
            return exact[0]

    return sorted(candidates)[0]


def get_zip_member_bytes(url, entry, chunk_size=1024 * 1024):
    # Read local file header to determine variable header length.
    offset = entry["local_header_offset"]

    header, _ = http_get_bytes(
        url,
        offset,
        offset + 30 - 1,
    )

    if header[:4] != b"PK\x03\x04":
        raise RuntimeError("Invalid local ZIP header.")

    (
        signature,
        version,
        flags,
        compression,
        mod_time,
        mod_date,
        crc32,
        comp_size32,
        uncomp_size32,
        name_len,
        extra_len,
    ) = struct.unpack("<4s5H3I2H", header)

    data_start = offset + 30 + name_len + extra_len
    comp_size = entry["compressed_size"]
    data_end = data_start + comp_size - 1

    chunks = []
    for start in range(data_start, data_end + 1, chunk_size):
        end = min(start + chunk_size - 1, data_end)
        chunk, _ = http_get_bytes(url, start, end)
        chunks.append(chunk)

    compressed = b"".join(chunks)

    if compression == 0:
        return compressed

    if compression == 8:
        import zlib
        return zlib.decompress(compressed, -15)

    raise RuntimeError(
        f"Unsupported ZIP compression method: {compression}"
    )


print("Patient ZIP-member download helper defined.")


Patient ZIP-member download helper defined.


In [79]:
# Cell 8 — Download/cache small annotation ZIP archives

def cache_small_file(url, destination):
    destination = Path(destination)
    if destination.exists():
        return destination

    data, _ = http_get_bytes(url)
    destination.write_bytes(data)
    return destination


def load_label_zip_entries(source_name):
    cfg = SOURCES[source_name]
    cache_path = CACHE_ROOT / cfg["label_zip"]

    cache_small_file(cfg["label_url"], cache_path)

    with zipfile.ZipFile(cache_path, "r") as zf:
        return zf.namelist()


for source_name in ["train_part1", "train_part2", "validation"]:
    print(source_name, "label archive:", SOURCES[source_name]["label_zip"])

print("Small annotation archives will be cached on demand.")


train_part1 label archive: ribfrac-train-labels-1.zip
train_part2 label archive: ribfrac-train-labels-2.zip
validation label archive: ribfrac-val-labels.zip
Small annotation archives will be cached on demand.


In [80]:

# ============================================================
# CELL #9 — RESTORE RIBFRAC METADATA WITHOUT ZENODO
# ============================================================


print("=" * 65)
print("RIBFRAC METADATA RECOVERY")
print("=" * 65)

# ============================================================
# PART A — CONFIGURE DATASET LOCATIONS
# ============================================================

PERSISTED_ROOT = Path(
    "/kaggle/input/datasets/waboke/"
    "ribfrac-preprocessed-final"
)

# Check for previously saved combined metadata
combined_candidates = [
    PERSISTED_ROOT / "metadata" /
    "combined_ribfrac_metadata.csv",

    METADATA_ROOT /
    "combined_ribfrac_metadata.csv",

    PERSISTED_ROOT /
    "combined_ribfrac_metadata.csv"
]

# ============================================================
# PART B — STANDARDIZE METADATA
# ============================================================

def validate_metadata(df, source_name):

    df = df.copy()

    required = {
        "public_id",
        "label_id",
        "label_code"
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{source_name}: missing columns {missing}"
        )

    if df[list(required)].isna().any().any():
        raise ValueError(
            f"{source_name}: missing annotation values"
        )

    df["public_id"] = (
        df["public_id"].astype(str)
    )

    df["label_id"] = pd.to_numeric(
        df["label_id"],
        errors="raise"
    )

    df["label_code"] = pd.to_numeric(
        df["label_code"],
        errors="raise"
    )

    # Prevent silent truncation of annotation values
    if not (
        (df["label_id"] % 1 == 0).all()
        and (df["label_code"] % 1 == 0).all()
    ):
        raise ValueError(
            f"{source_name}: non-integer metadata values"
        )

    df["label_id"] = df["label_id"].astype(int)
    df["label_code"] = df["label_code"].astype(int)

    df["source"] = source_name

    expected = SOURCES[source_name]["expected_patients"]
    actual = df["public_id"].nunique()

    if actual != expected:
        raise ValueError(
            f"{source_name}: expected {expected} "
            f"patients, found {actual}"
        )

    return df


# ============================================================
# PART C — LOCATE COMBINED METADATA
# ============================================================

combined_path = next(
    (p for p in combined_candidates if p.is_file()),
    None
)

source_metadata = {}

if combined_path is not None:

    print("\nFound existing combined metadata:")
    print(combined_path)

    combined_df = pd.read_csv(combined_path)

    # Support the naming conventions of Notebook 01
    # and Notebook 02.
    if (
        "source_partition" in combined_df.columns
        and "source" not in combined_df.columns
    ):
        combined_df = combined_df.rename(
            columns={"source_partition": "source"}
        )

    if "source" not in combined_df.columns:
        raise ValueError(
            "Combined metadata does not contain "
            "a source/source_partition column."
        )

    # Separate the three original source cohorts
    for source_name in SOURCES:

        rows = combined_df.loc[
            combined_df["source"] == source_name
        ].copy()

        source_metadata[source_name] = (
            validate_metadata(rows, source_name)
        )

else:

    print("\nCombined metadata not found.")
    print("Searching for individual source CSV files.")

    # --------------------------------------------------------
    # PART D — USE INDIVIDUAL CACHED METADATA
    # --------------------------------------------------------

    for source_name, cfg in SOURCES.items():

        filename = cfg["info_filename"]

        candidates = [
            CACHE_ROOT / filename,
            METADATA_ROOT / filename,
            PERSISTED_ROOT / "metadata" / filename,
            PERSISTED_ROOT / "cache" / filename
        ]

        metadata_path = next(
            (p for p in candidates if p.is_file()),
            None
        )

        if metadata_path is None:
            raise FileNotFoundError(
                f"No local metadata for {source_name}.\n"
                f"Expected file: {filename}\n"
                "Attach a Kaggle dataset containing "
                "the original metadata or the "
                "combined_ribfrac_metadata.csv file."
            )

        print(f"\n{source_name}: {metadata_path}")

        df = pd.read_csv(metadata_path)

        source_metadata[source_name] = (
            validate_metadata(df, source_name)
        )


# ============================================================
# PART E — VERIFY ALL THREE SOURCE COHORTS
# ============================================================

print("\n" + "-" * 65)
print("SOURCE METADATA SUMMARY")
print("-" * 65)

for source_name in SOURCES:

    df = source_metadata[source_name]

    print(
        f"{source_name:15s} | "
        f"rows: {len(df):5d} | "
        f"patients: {df['public_id'].nunique():3d}"
    )

# Verify no patient is shared across source cohorts
patient_sets = [
    set(df["public_id"])
    for df in source_metadata.values()
]

all_ids = set().union(*patient_sets)

assert sum(map(len, patient_sets)) == 500
assert len(all_ids) == 500

print("\nTotal unique patients:", len(all_ids))

# ============================================================
# PART F — FINAL VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("CELL #9 VALIDATION: PASS")
print("=" * 65)

print("Metadata loaded from existing local files.")
print("No Zenodo download was required.")


RIBFRAC METADATA RECOVERY

Found existing combined metadata:
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/metadata/combined_ribfrac_metadata.csv

-----------------------------------------------------------------
SOURCE METADATA SUMMARY
-----------------------------------------------------------------
train_part1     | rows:  3152 | patients: 300
train_part2     | rows:  1255 | patients: 120
validation      | rows:   515 | patients:  80

Total unique patients: 500

CELL #9 VALIDATION: PASS
Metadata loaded from existing local files.
No Zenodo download was required.


##  10 — Verify the three source cohorts

In [81]:
# Cell 10 — Verify the three source cohorts


# ============================================================
# CELL #10 — VERIFY THE THREE RIBFRAC SOURCE COHORTS
# ============================================================

print("=" * 65)
print("RIBFRAC SOURCE COHORT VERIFICATION")
print("=" * 65)

# ============================================================
# PART A — VERIFY REQUIRED SOURCE DEFINITIONS
# ============================================================

expected_sources = {
    "train_part1": 300,
    "train_part2": 120,
    "validation": 80
}

assert set(SOURCES.keys()) == set(expected_sources), (
    "Unexpected RibFrac source configuration."
)

assert set(source_metadata.keys()) == set(expected_sources), (
    "Metadata is missing one or more source cohorts."
)

for source_name, expected_count in expected_sources.items():

    assert (
        SOURCES[source_name]["expected_patients"]
        == expected_count
    )

print("\nAll three source definitions: PASS")


# ============================================================
# PART B — VERIFY METADATA STRUCTURE
# ============================================================

required_columns = {
    "public_id",
    "label_id",
    "label_code",
    "source"
}

cohort_summary_records = []

for source_name, expected_count in expected_sources.items():

    print("\n" + "-" * 65)
    print("VERIFYING SOURCE:", source_name)
    print("-" * 65)

    df = source_metadata[source_name]

    # --------------------------------------------------------
    # 1. Verify required metadata columns
    # --------------------------------------------------------

    missing_columns = required_columns - set(df.columns)

    assert not missing_columns, (
        f"{source_name}: Missing columns "
        f"{missing_columns}"
    )

    assert len(df) > 0, (
        f"{source_name}: Empty metadata table."
    )

    # --------------------------------------------------------
    # 2. Verify no missing metadata values
    # --------------------------------------------------------

    assert df[list(required_columns)].notna().all().all(), (
        f"{source_name}: Missing metadata values."
    )

    assert df["public_id"].astype(str).str.strip().ne("").all(), (
        f"{source_name}: Empty patient IDs."
    )

    # --------------------------------------------------------
    # 3. Verify patient count
    # --------------------------------------------------------

    unique_patients = df["public_id"].nunique()

    assert unique_patients == expected_count, (
        f"{source_name}: Expected {expected_count} "
        f"patients, found {unique_patients}"
    )

    # --------------------------------------------------------
    # 4. Verify source assignment
    # --------------------------------------------------------

    assert (
        df["source"] == source_name
    ).all(), (
        f"{source_name}: Incorrect source assignments."
    )

    # --------------------------------------------------------
    # 5. Verify fracture annotation IDs
    # --------------------------------------------------------

    label_ids = pd.to_numeric(
        df["label_id"],
        errors="raise"
    )

    label_codes = pd.to_numeric(
        df["label_code"],
        errors="raise"
    )

    assert np.isfinite(label_ids.to_numpy()).all()
    assert np.isfinite(label_codes.to_numpy()).all()

    assert (label_ids >= 0).all(), (
        f"{source_name}: Negative annotation IDs found."
    )

    assert (label_ids % 1 == 0).all(), (
        f"{source_name}: Non-integer annotation IDs."
    )

    assert (label_codes % 1 == 0).all(), (
        f"{source_name}: Non-integer subtype codes."
    )

    # One metadata entry per patient and annotation ID.
    duplicate_annotations = df.duplicated(
        subset=["public_id", "label_id"]
    ).sum()

    assert duplicate_annotations == 0, (
        f"{source_name}: "
        f"{duplicate_annotations} duplicate "
        f"patient/annotation ID records."
    )

    # --------------------------------------------------------
    # 6. Summarize fracture annotations
    # --------------------------------------------------------

    fracture_rows = df.loc[
        label_ids > 0
    ].copy()

    fracture_count = len(fracture_rows)

    unknown_count = int(
        (fracture_rows["label_code"] == -1).sum()
    )

    # Do not discard unknown subtype fractures.
    classified_count = fracture_count - unknown_count

    # --------------------------------------------------------
    # 7. Store cohort summary
    # --------------------------------------------------------

    cohort_summary_records.append({
        "source": source_name,
        "expected_patients": expected_count,
        "actual_patients": unique_patients,
        "metadata_rows": len(df),
        "fracture_annotations": fracture_count,
        "unknown_subtypes": unknown_count,
        "other_subtype_records": classified_count,
        "duplicate_annotations": int(
            duplicate_annotations
        ),
        "status": "PASS"
    })

    print("Expected patients:", expected_count)
    print("Unique patients:", unique_patients)
    print("Metadata records:", len(df))
    print("Positive annotation IDs:", fracture_count)
    print("Unknown subtype records:", unknown_count)
    print("Duplicate annotation records:", duplicate_annotations)

    print("\nObserved subtype codes:")
    print(
        fracture_rows["label_code"]
        .value_counts()
        .sort_index()
    )

    print("\nSOURCE VERIFICATION: PASS")


# ============================================================
# PART C — CREATE THREE-SOURCE SUMMARY
# ============================================================

source_cohort_summary = pd.DataFrame(
    cohort_summary_records
)

print("\n" + "=" * 65)
print("RIBFRAC SOURCE COHORT SUMMARY")
print("=" * 65)

display(source_cohort_summary)


# ============================================================
# PART D — VERIFY TOTAL PATIENT COUNT
# ============================================================

total_expected = int(
    source_cohort_summary["expected_patients"].sum()
)

total_actual = int(
    source_cohort_summary["actual_patients"].sum()
)

assert total_expected == 500
assert total_actual == 500

print("\nTotal expected cohort entries:", total_expected)
print("Total verified cohort entries:", total_actual)

# Individual patient overlap across sources
# will be verified separately in Cell #11.

# ============================================================
# PART E — FINAL VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("CELL #10 — FINAL VALIDATION")
print("=" * 65)

for source_name in expected_sources:
    print(f"{source_name}: PASS")

print("\nExpected cohort total: 500")
print("Verified cohort total:", total_actual)

print("\nCELL #10 VALIDATION: PASS")
print("All three source metadata cohorts verified.")


RIBFRAC SOURCE COHORT VERIFICATION

All three source definitions: PASS

-----------------------------------------------------------------
VERIFYING SOURCE: train_part1
-----------------------------------------------------------------
Expected patients: 300
Unique patients: 300
Metadata records: 3152
Positive annotation IDs: 2852
Unknown subtype records: 1646
Duplicate annotation records: 0

Observed subtype codes:
label_code
-1    1646
 1     450
 2     417
 3     197
 4     142
Name: count, dtype: int64

SOURCE VERIFICATION: PASS

-----------------------------------------------------------------
VERIFYING SOURCE: train_part2
-----------------------------------------------------------------
Expected patients: 120
Unique patients: 120
Metadata records: 1255
Positive annotation IDs: 1135
Unknown subtype records: 686
Duplicate annotation records: 0

Observed subtype codes:
label_code
-1    686
 1    168
 2    150
 3     94
 4     37
Name: count, dtype: int64

SOURCE VERIFICATION: PASS

--

,source,expected_patients,actual_patients,metadata_rows,fracture_annotations,unknown_subtypes,other_subtype_records,duplicate_annotations,status
0,train_part1,300,300,3152,2852,1646,1206,0,PASS
1,train_part2,120,120,1255,1135,686,449,0,PASS
2,validation,80,80,515,435,243,192,0,PASS



Total expected cohort entries: 500
Total verified cohort entries: 500

CELL #10 — FINAL VALIDATION
train_part1: PASS
train_part2: PASS
validation: PASS

Expected cohort total: 500
Verified cohort total: 500

CELL #10 VALIDATION: PASS
All three source metadata cohorts verified.


## 11 — Verify patient uniqueness across the three sources

In [82]:

# ============================================================
# CELL #11 — VERIFY PATIENT UNIQUENESS ACROSS SOURCES
# ============================================================

import pandas as pd
from itertools import combinations

print("=" * 65)
print("RIBFRAC CROSS-SOURCE PATIENT UNIQUENESS")
print("=" * 65)

# ============================================================
# PART A — VERIFY SOURCE METADATA
# ============================================================

expected_sources = {
    "train_part1": 300,
    "train_part2": 120,
    "validation": 80
}

assert set(source_metadata.keys()) == set(expected_sources), (
    "One or more source cohorts are missing."
)

# ============================================================
# PART B — EXTRACT UNIQUE PATIENT IDS
# ============================================================

patient_sets = {}

source_summary_records = []

for source_name, expected_count in expected_sources.items():

    df = source_metadata[source_name]

    assert "public_id" in df.columns

    patient_column = df["public_id"].astype(str).str.strip()

    assert patient_column.notna().all()
    assert patient_column.ne("").all()

    unique_ids = set(patient_column)

    patient_sets[source_name] = unique_ids

    actual_count = len(unique_ids)

    assert actual_count == expected_count, (
        f"{source_name}: expected {expected_count}, "
        f"found {actual_count}"
    )

    source_summary_records.append({
        "source": source_name,
        "metadata_records": len(df),
        "unique_patients": actual_count,
        "expected_patients": expected_count,
        "status": "PASS"
    })

    print(
        f"{source_name}: "
        f"{actual_count} unique patients"
    )

source_uniqueness_summary = pd.DataFrame(
    source_summary_records
)

print("\nSource cohort summary:")
display(source_uniqueness_summary)

# ============================================================
# PART C — CHECK PAIRWISE SOURCE OVERLAP
# ============================================================

print("\n" + "-" * 65)
print("PAIRWISE PATIENT OVERLAP")
print("-" * 65)

overlap_records = []

source_names = list(expected_sources.keys())

for source_a, source_b in combinations(source_names, 2):

    overlap = patient_sets[source_a] & patient_sets[source_b]

    overlap_count = len(overlap)

    overlap_records.append({
        "source_a": source_a,
        "source_b": source_b,
        "overlapping_patients": overlap_count,
        "status": (
            "PASS" if overlap_count == 0 else "FAIL"
        )
    })

    print(
        f"{source_a} / {source_b}: "
        f"{overlap_count} overlapping patients"
    )

    if overlap:
        print("Overlapping patient IDs:")
        print(sorted(overlap)[:20])

overlap_summary_df = pd.DataFrame(
    overlap_records
)

print("\nCross-source overlap summary:")
display(overlap_summary_df)

# Stop if any overlap exists.
assert (
    overlap_summary_df["overlapping_patients"] == 0
).all(), (
    "Patient identifiers overlap across source cohorts."
)

print("\nPairwise patient overlap verification: PASS")

# ============================================================
# PART D — CREATE THE COMBINED PATIENT ID REGISTRY
# ============================================================

# Preserve the variable used in Cell #12 and Cell #13.

combined_patient_ids = set().union(
    *patient_sets.values()
)

total_source_patients = sum(
    len(ids) for ids in patient_sets.values()
)

TOTAL_EXPECTED = sum(
    expected_sources.values()
)

assert TOTAL_EXPECTED == 500

assert total_source_patients == 500

assert len(combined_patient_ids) == 500, (
    "Combined cohort does not contain "
    "500 unique patient identifiers."
)

print("\n" + "-" * 65)
print("COMBINED PATIENT REGISTRY")
print("-" * 65)

print("Total expected patients:", TOTAL_EXPECTED)
print("Total source patient counts:", total_source_patients)
print("Combined unique patients:", len(combined_patient_ids))

# ============================================================
# PART E — VERIFY UNIQUE SOURCE ASSIGNMENT
# ============================================================

patient_source_records = []

for source_name, ids in patient_sets.items():

    for patient_id in sorted(ids):

        patient_source_records.append({
            "public_id": patient_id,
            "source": source_name
        })

patient_source_check = pd.DataFrame(
    patient_source_records
)

assert len(patient_source_check) == 500

assert not patient_source_check[
    "public_id"
].duplicated().any()

assert patient_source_check[
    "public_id"
].nunique() == 500

assert set(
    patient_source_check["public_id"]
) == combined_patient_ids

print("\nOne-to-one patient/source assignment: PASS")

print("\nFirst 10 patient registry entries:")
display(
    patient_source_check
    .sort_values("public_id")
    .head(10)
)

# ============================================================
# PART F — FINAL VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("CELL #11 — FINAL VALIDATION")
print("=" * 65)

print("Training Part 1:", len(patient_sets["train_part1"]))
print("Training Part 2:", len(patient_sets["train_part2"]))
print("Original Validation:", len(patient_sets["validation"]))

print("\nTotal unique patients:", len(combined_patient_ids))
print("Cross-source overlapping patient IDs: 0")

print("\nCELL #11 VALIDATION: PASS")
print("All 500 patient identifiers are unique across sources.")


RIBFRAC CROSS-SOURCE PATIENT UNIQUENESS
train_part1: 300 unique patients
train_part2: 120 unique patients
validation: 80 unique patients

Source cohort summary:


,source,metadata_records,unique_patients,expected_patients,status
0,train_part1,3152,300,300,PASS
1,train_part2,1255,120,120,PASS
2,validation,515,80,80,PASS



-----------------------------------------------------------------
PAIRWISE PATIENT OVERLAP
-----------------------------------------------------------------
train_part1 / train_part2: 0 overlapping patients
train_part1 / validation: 0 overlapping patients
train_part2 / validation: 0 overlapping patients

Cross-source overlap summary:


,source_a,source_b,overlapping_patients,status
0,train_part1,train_part2,0,PASS
1,train_part1,validation,0,PASS
2,train_part2,validation,0,PASS



Pairwise patient overlap verification: PASS

-----------------------------------------------------------------
COMBINED PATIENT REGISTRY
-----------------------------------------------------------------
Total expected patients: 500
Total source patient counts: 500
Combined unique patients: 500

One-to-one patient/source assignment: PASS

First 10 patient registry entries:


,public_id,source
0,RibFrac1,train_part1
1,RibFrac10,train_part1
2,RibFrac100,train_part1
3,RibFrac101,train_part1
4,RibFrac102,train_part1
5,RibFrac103,train_part1
6,RibFrac104,train_part1
7,RibFrac105,train_part1
8,RibFrac106,train_part1
9,RibFrac107,train_part1



CELL #11 — FINAL VALIDATION
Training Part 1: 300
Training Part 2: 120
Original Validation: 80

Total unique patients: 500
Cross-source overlapping patient IDs: 0

CELL #11 VALIDATION: PASS
All 500 patient identifiers are unique across sources.


In [83]:

# ============================================================
# CELL #12 — COMBINE RIBFRAC METADATA FROM THREE SOURCES
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 65)
print("RIBFRAC — COMBINE SOURCE METADATA")
print("=" * 65)

# ============================================================
# PART A — DEFINE EXPECTED SOURCES
# ============================================================

expected_sources = {
    "train_part1": 300,
    "train_part2": 120,
    "validation": 80
}

required_columns = [
    "public_id",
    "label_id",
    "label_code",
    "source"
]

assert set(source_metadata) == set(expected_sources), (
    "One or more RibFrac metadata sources are missing."
)

# ============================================================
# PART B — STANDARDIZE SOURCE METADATA
# ============================================================

metadata_tables = []

for source_name, expected_count in expected_sources.items():

    df = source_metadata[source_name].copy()

    # Support the source naming convention from Notebook 01.
    if (
        "source_partition" in df.columns
        and "source" not in df.columns
    ):
        df = df.rename(
            columns={"source_partition": "source"}
        )

    missing = set(required_columns) - set(df.columns)

    if missing:
        raise ValueError(
            f"{source_name}: missing columns {missing}"
        )

    df = df[required_columns].copy()

    if df.isna().any().any():
        raise ValueError(
            f"{source_name}: missing metadata values."
        )

    df["public_id"] = (
        df["public_id"].astype(str).str.strip()
    )

    if df["public_id"].eq("").any():
        raise ValueError(
            f"{source_name}: empty patient identifiers."
        )

    for column in ["label_id", "label_code"]:

        values = pd.to_numeric(
            df[column],
            errors="raise"
        )

        if not np.isfinite(values.to_numpy()).all():
            raise ValueError(
                f"{source_name}: invalid {column} values."
            )

        if not (values % 1 == 0).all():
            raise ValueError(
                f"{source_name}: non-integer {column} values."
            )

        df[column] = values.astype(np.int64)

    if (df["label_id"] < 0).any():
        raise ValueError(
            f"{source_name}: negative annotation IDs."
        )

    if not (df["source"] == source_name).all():
        raise ValueError(
            f"{source_name}: incorrect source assignments."
        )

    actual_patients = df["public_id"].nunique()

    assert actual_patients == expected_count, (
        f"{source_name}: expected {expected_count} "
        f"patients, found {actual_patients}."
    )

    # Annotation instance IDs must be unique within
    # each patient's metadata.
    duplicates = df.duplicated(
        subset=["public_id", "label_id"]
    )

    if duplicates.any():
        raise ValueError(
            f"{source_name}: duplicate patient/label IDs."
        )

    metadata_tables.append(df)

    print(
        f"{source_name:15s}: "
        f"{actual_patients:3d} patients, "
        f"{len(df):5d} annotation records"
    )

# ============================================================
# PART C — COMBINE THE THREE SOURCE TABLES
# ============================================================

combined_metadata = pd.concat(
    metadata_tables,
    ignore_index=True
)

combined_metadata = (
    combined_metadata
    .sort_values(
        ["public_id", "label_id", "source"]
    )
    .reset_index(drop=True)
)

print("\nCombined metadata shape:")
print(combined_metadata.shape)

# ============================================================
# PART D — VERIFY COMPLETE PATIENT COVERAGE
# ============================================================

combined_ids = set(
    combined_metadata["public_id"]
)

assert combined_ids == set(combined_patient_ids), (
    "Combined metadata does not match the "
    "verified patient registry from Cell #11."
)

assert len(combined_ids) == 500

source_counts = (
    combined_metadata
    .groupby("source")["public_id"]
    .nunique()
    .to_dict()
)

assert source_counts == expected_sources, (
    "Incorrect patient counts after metadata merging."
)

print("\nPatient coverage: PASS")
print("Total unique patients:", len(combined_ids))

# ============================================================
# PART E — VERIFY NO DUPLICATE ANNOTATIONS
# ============================================================

duplicate_count = int(
    combined_metadata.duplicated(
        subset=["public_id", "label_id"]
    ).sum()
)

assert duplicate_count == 0, (
    "Duplicate patient/annotation ID combinations found."
)

# Verify each patient belongs to one source
sources_per_patient = (
    combined_metadata
    .groupby("public_id")["source"]
    .nunique()
)

assert (sources_per_patient == 1).all(), (
    "Some patients belong to multiple sources."
)

print("Duplicate annotation records:", duplicate_count)
print("Patient/source uniqueness: PASS")

# ============================================================
# PART F — SUMMARIZE FRACTURE ANNOTATIONS
# ============================================================

# Annotation ID > 0 identifies an annotated fracture
# instance. Background ID 0 is not counted as a fracture.

fracture_records = combined_metadata.loc[
    combined_metadata["label_id"] > 0
].copy()

unknown_subtypes = fracture_records.loc[
    fracture_records["label_code"] == -1
]

print("\n" + "-" * 65)
print("COMBINED FRACTURE ANNOTATION SUMMARY")
print("-" * 65)

print("Total metadata records:", len(combined_metadata))
print("Positive fracture-instance records:", len(fracture_records))
print("Unknown subtype records:", len(unknown_subtypes))

print("\nFracture subtype distribution:")
print(
    fracture_records["label_code"]
    .value_counts()
    .sort_index()
)

# ============================================================
# PART G — DISPLAY SOURCE SUMMARY
# ============================================================

source_summary = (
    combined_metadata
    .groupby("source")
    .agg(
        metadata_records=("public_id", "size"),
        unique_patients=("public_id", "nunique")
    )
    .reset_index()
)

fracture_counts = (
    fracture_records
    .groupby("source")
    .size()
    .rename("fracture_annotations")
)

source_summary = source_summary.merge(
    fracture_counts,
    on="source",
    how="left"
)

source_summary["fracture_annotations"] = (
    source_summary["fracture_annotations"]
    .fillna(0)
    .astype(int)
)

print("\nSource metadata summary:")
display(source_summary)

# ============================================================
# PART H — SAVE COMBINED METADATA
# ============================================================

COMBINED_METADATA_PATH = (
    METADATA_ROOT / "combined_ribfrac_metadata.csv"
)

COMBINED_METADATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Preserve an existing identical file.
# Never silently overwrite conflicting metadata.

if COMBINED_METADATA_PATH.is_file():

    existing = pd.read_csv(COMBINED_METADATA_PATH)

    if (
        "source_partition" in existing.columns
        and "source" not in existing.columns
    ):
        existing = existing.rename(
            columns={"source_partition": "source"}
        )

    existing = existing[required_columns].copy()

    existing["public_id"] = (
        existing["public_id"].astype(str)
    )

    for column in ["label_id", "label_code"]:
        existing[column] = pd.to_numeric(
            existing[column],
            errors="raise"
        ).astype(np.int64)

    existing = (
        existing
        .sort_values(
            ["public_id", "label_id", "source"]
        )
        .reset_index(drop=True)
    )

    if not existing.equals(combined_metadata):
        raise RuntimeError(
            "Existing combined metadata differs from "
            "the verified source records. "
            "No file was overwritten."
        )

    print("\nExisting combined metadata verified.")
    print("No overwrite necessary.")

else:

    combined_metadata.to_csv(
        COMBINED_METADATA_PATH,
        index=False
    )

    print("\nCombined metadata saved successfully.")

# ============================================================
# PART I — FINAL VALIDATION
# ============================================================

saved_metadata = pd.read_csv(
    COMBINED_METADATA_PATH
)

assert saved_metadata["public_id"].nunique() == 500

assert len(saved_metadata) == len(combined_metadata)

assert set(saved_metadata.columns) == set(required_columns)

print("\n" + "=" * 65)
print("CELL #12 — FINAL VALIDATION")
print("=" * 65)

print("Training Part 1 patients:", source_counts["train_part1"])
print("Training Part 2 patients:", source_counts["train_part2"])
print("Original Validation patients:", source_counts["validation"])

print("\nTotal unique patients:", len(combined_ids))
print("Total annotation records:", len(combined_metadata))
print("Duplicate annotation IDs:", duplicate_count)

print("\nSaved file:")
print(COMBINED_METADATA_PATH)

print("\nFirst 10 combined records:")
display(combined_metadata.head(10))

print("\nCELL #12 VALIDATION: PASS")
print("Combined metadata verified successfully.")


RIBFRAC — COMBINE SOURCE METADATA
train_part1    : 300 patients,  3152 annotation records
train_part2    : 120 patients,  1255 annotation records
validation     :  80 patients,   515 annotation records

Combined metadata shape:
(4922, 4)

Patient coverage: PASS
Total unique patients: 500
Duplicate annotation records: 0
Patient/source uniqueness: PASS

-----------------------------------------------------------------
COMBINED FRACTURE ANNOTATION SUMMARY
-----------------------------------------------------------------
Total metadata records: 4922
Positive fracture-instance records: 4422
Unknown subtype records: 2575

Fracture subtype distribution:
label_code
-1    2575
 1     687
 2     630
 3     321
 4     209
Name: count, dtype: int64

Source metadata summary:


,source,metadata_records,unique_patients,fracture_annotations
0,train_part1,3152,300,2852
1,train_part2,1255,120,1135
2,validation,515,80,435



Existing combined metadata verified.
No overwrite necessary.

CELL #12 — FINAL VALIDATION
Training Part 1 patients: 300
Training Part 2 patients: 120
Original Validation patients: 80

Total unique patients: 500
Total annotation records: 4922
Duplicate annotation IDs: 0

Saved file:
/kaggle/working/ribfrac_preprocessed_final/metadata/combined_ribfrac_metadata.csv

First 10 combined records:


,public_id,label_id,label_code,source
0,RibFrac1,0,0,train_part1
1,RibFrac1,1,2,train_part1
2,RibFrac1,2,2,train_part1
3,RibFrac10,0,0,train_part1
4,RibFrac10,1,-1,train_part1
5,RibFrac10,2,-1,train_part1
6,RibFrac10,3,3,train_part1
7,RibFrac10,4,-1,train_part1
8,RibFrac10,5,3,train_part1
9,RibFrac10,6,3,train_part1



CELL #12 VALIDATION: PASS
Combined metadata verified successfully.


## Final patient-level split

The split is created **after combining all 500 labelled cases**. This prevents the Part-1-only split used during development from being accidentally reused.

The split is:
- 70% training = 350 patients
- 15% validation = 75 patients
- 15% internal test = 75 patients

A fixed seed of 42 is used for reproducibility.


## 13 — Load the Authoritative Patient Split

In [84]:

# ============================================================
# CELL #13 — LOAD THE AUTHORITATIVE PATIENT SPLIT
# ============================================================

# 1. Reference dataset from Notebook 01
PERSISTED_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

REFERENCE_SPLIT_PATH = (
    PERSISTED_ROOT / "metadata/patient_split_500.csv"
)

SPLIT_PATH = METADATA_ROOT / "patient_split_500.csv"

if not REFERENCE_SPLIT_PATH.is_file():
    raise FileNotFoundError(
        f"Authoritative split not found: {REFERENCE_SPLIT_PATH}\n"
        "Attach the persisted RibFrac dataset to Kaggle."
    )

# 2. Load and validate the patient split
def load_verified_patient_split(path):

    df = pd.read_csv(path, dtype=str)

    if "public_id" not in df.columns:
        if "patient_id" in df.columns:
            df = df.rename(
                columns={"patient_id": "public_id"}
            )

    required = {"public_id", "split"}

    if not required.issubset(df.columns):
        raise ValueError("Required columns are missing.")

    df = df[["public_id", "split"]].copy()

    if df.isna().any().any():
        raise ValueError("Missing patient IDs or split labels.")

    df["public_id"] = df["public_id"].str.strip()
    df["split"] = (
        df["split"].str.strip().str.lower()
        .replace({"val": "validation"})
    )

    assert df["public_id"].ne("").all()
    assert len(df) == 500
    assert df["public_id"].nunique() == 500

    expected = {
        "train": 350,
        "validation": 75,
        "test": 75
    }

    assert df["split"].value_counts().to_dict() == expected

    return df.sort_values("public_id").reset_index(drop=True)


patient_split = load_verified_patient_split(
    REFERENCE_SPLIT_PATH
)

# 3. Verify the split against Cell #12 metadata
if "combined_metadata" not in globals():
    raise RuntimeError("Execute Cell #12 first.")

metadata_ids = set(
    combined_metadata["public_id"].astype(str).str.strip()
)

split_ids = set(patient_split["public_id"])

assert len(metadata_ids) == 500
assert metadata_ids == split_ids

# 4. Restore patient arrays
all_patients = np.array(sorted(split_ids))

train_patients = patient_split.loc[
    patient_split["split"] == "train", "public_id"
].to_numpy()

val_patients = patient_split.loc[
    patient_split["split"] == "validation", "public_id"
].to_numpy()

test_patients = patient_split.loc[
    patient_split["split"] == "test", "public_id"
].to_numpy()

# 5. Check patient-level independence
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

assert not (train_set & val_set)
assert not (train_set & test_set)
assert not (val_set & test_set)

assert len(train_set | val_set | test_set) == 500

# 6. Preserve the authoritative assignments
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

if SPLIT_PATH.is_file():

    existing = load_verified_patient_split(SPLIT_PATH)

    if not existing.equals(patient_split):
        raise RuntimeError(
            "Existing split differs from authoritative split. "
            "No overwrite performed."
        )

else:
    patient_split.to_csv(SPLIT_PATH, index=False)

# 7. Verify the saved working copy
saved_split = load_verified_patient_split(SPLIT_PATH)
assert saved_split.equals(patient_split)

# 8. Check previously persisted CT-label pairs
image_ids = {
    p.name.removesuffix("-image.npy")
    for p in (PERSISTED_ROOT / "images").glob("*-image.npy")
}

label_ids = {
    p.name.removesuffix("-label.npy")
    for p in (PERSISTED_ROOT / "labels").glob("*-label.npy")
}

complete_ids = image_ids & label_ids

assert complete_ids.issubset(split_ids)

# 9. Final report
print("=" * 60)
print("CELL #13 — AUTHORITATIVE PATIENT SPLIT")
print("=" * 60)

print("Total patients:", len(patient_split))
print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Testing patients:", len(test_patients))

print("\nPatient overlap: 0")
print("Complete persisted CT-label pairs:", len(complete_ids))

print("\nReference:", REFERENCE_SPLIT_PATH)
print("Working copy:", SPLIT_PATH)

print("\nCELL #13 VALIDATION: PASS")
print("Existing patient assignments preserved.")

display(patient_split.head(10))


CELL #13 — AUTHORITATIVE PATIENT SPLIT
Total patients: 500
Training patients: 350
Validation patients: 75
Testing patients: 75

Patient overlap: 0
Complete persisted CT-label pairs: 13

Reference: /kaggle/input/datasets/waboke/ribfrac-preprocessed-final/metadata/patient_split_500.csv
Working copy: /kaggle/working/ribfrac_preprocessed_final/metadata/patient_split_500.csv

CELL #13 VALIDATION: PASS
Existing patient assignments preserved.


,public_id,split
0,RibFrac1,validation
1,RibFrac10,train
2,RibFrac100,validation
3,RibFrac101,train
4,RibFrac102,train
5,RibFrac103,train
6,RibFrac104,train
7,RibFrac105,train
8,RibFrac106,train
9,RibFrac107,test


## 14 — CREATE PATIENT SOURCE REGISTRY

In [85]:
# Cell 14 — Create patient source registry

source_lookup = (
    combined_metadata[["public_id", "source"]]
    .drop_duplicates()
)

assert not source_lookup["public_id"].duplicated().any()

patient_registry = (
    patient_split
    .merge(source_lookup, on="public_id", how="left", validate="one_to_one")
)

assert patient_registry["source"].notna().all()
assert len(patient_registry) == 500

REGISTRY_PATH = METADATA_ROOT / "patient_source_registry_500.csv"
patient_registry.to_csv(REGISTRY_PATH, index=False)

print(patient_registry.groupby(["source", "split"]).size())
print("Saved:", REGISTRY_PATH)


source       split     
train_part1  test           51
             train         207
             validation     42
train_part2  test           15
             train          85
             validation     20
validation   test            9
             train          58
             validation     13
dtype: int64
Saved: /kaggle/working/ribfrac_preprocessed_final/metadata/patient_source_registry_500.csv


## Cell 15 — Verify final source/split counts


In [86]:
# Cell 15 — Verify final source/split counts

summary = (
    patient_registry
    .groupby(["source", "split"])
    .size()
    .unstack(fill_value=0)
)

print(summary)

print("\nTotal by source:")
print(patient_registry["source"].value_counts())

print("\nTotal by split:")
print(patient_registry["split"].value_counts())

assert patient_registry["public_id"].nunique() == 500
assert patient_registry["split"].value_counts().to_dict() == {
    "train": 350,
    "validation": 75,
    "test": 75,
}

print("\nFinal 500-patient registry validation: PASS")


split        test  train  validation
source                              
train_part1    51    207          42
train_part2    15     85          20
validation      9     58          13

Total by source:
source
train_part1    300
train_part2    120
validation      80
Name: count, dtype: int64

Total by split:
split
train         350
validation     75
test           75
Name: count, dtype: int64

Final 500-patient registry validation: PASS


## Preprocessing configuration

The validated development preprocessing is retained:

- Target spacing: `(1.0, 1.0, 1.0)` mm
- CT interpolation: linear/trilinear
- Label interpolation: nearest-neighbor
- HU clipping: `[-1000, 400]`
- CT normalization: `[0, 1]`
- CT dtype: `float32`
- Label dtype: `int16`

Labels remain **annotation IDs** at this stage. Notebook 03 maps annotation IDs to fracture classes using the combined metadata.


## 16 — Final preprocessing configuration

In [88]:


# Cell 16 — Final preprocessing configuration

TARGET_SPACING = np.array([1.0, 1.0, 1.0], dtype=np.float32)

HU_MIN = -1000.0
HU_MAX = 400.0

CT_DTYPE = np.float32
LABEL_DTYPE = np.int16

PREPROCESSING_CONFIG = {
    "target_spacing_mm": [1.0, 1.0, 1.0],
    "ct_interpolation": "linear",
    "label_interpolation": "nearest_neighbor",
    "hu_clip": [-1000.0, 400.0],
    "ct_normalization": "[0,1]",
    "ct_dtype": "float32",
    "label_dtype": "int16",
    "seed": SEED,
    "cohort_size": 500,
    "split": {
        "train": 350,
        "validation": 75,
        "test": 75,
    },
}

CONFIG_PATH = METADATA_ROOT / "preprocessing_config_500.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(PREPROCESSING_CONFIG, f, indent=2)

print(json.dumps(PREPROCESSING_CONFIG, indent=2))
print("Saved:", CONFIG_PATH)


{
  "target_spacing_mm": [
    1.0,
    1.0,
    1.0
  ],
  "ct_interpolation": "linear",
  "label_interpolation": "nearest_neighbor",
  "hu_clip": [
    -1000.0,
    400.0
  ],
  "ct_normalization": "[0,1]",
  "ct_dtype": "float32",
  "label_dtype": "int16",
  "seed": 42,
  "cohort_size": 500,
  "split": {
    "train": 350,
    "validation": 75,
    "test": 75
  }
}
Saved: /kaggle/working/ribfrac_preprocessed_final/metadata/preprocessing_config_500.json


##  17 — NIfTI preprocessing functions

In [89]:
# Cell 17 — NIfTI preprocessing functions

def normalize_ct_hu(ct):
    ct = np.asarray(ct, dtype=np.float32)
    ct = np.clip(ct, HU_MIN, HU_MAX)
    ct = (ct - HU_MIN) / (HU_MAX - HU_MIN)
    return ct.astype(CT_DTYPE, copy=False)


def resample_volume(volume, original_spacing, target_spacing, order):
    original_spacing = np.asarray(original_spacing, dtype=np.float32)
    target_spacing = np.asarray(target_spacing, dtype=np.float32)

    zoom_factors = original_spacing / target_spacing

    resampled = zoom(
        volume,
        zoom=zoom_factors,
        order=order,
        mode="nearest",
        prefilter=(order != 0),
    )

    return resampled


def preprocess_nifti_pair(ct_nii, label_nii):
    ct = ct_nii.get_fdata(dtype=np.float32)
    label = label_nii.get_fdata(dtype=np.float32)

    ct_spacing = np.asarray(ct_nii.header.get_zooms()[:3], dtype=np.float32)
    label_spacing = np.asarray(label_nii.header.get_zooms()[:3], dtype=np.float32)

    if not np.allclose(ct_spacing, label_spacing, atol=1e-4):
        raise ValueError(
            f"CT/label spacing mismatch: {ct_spacing} vs {label_spacing}"
        )

    if ct.shape != label.shape:
        raise ValueError(
            f"CT/label shape mismatch: {ct.shape} vs {label.shape}"
        )

    ct = normalize_ct_hu(ct)

    ct_resampled = resample_volume(
        ct,
        ct_spacing,
        TARGET_SPACING,
        order=1,
    ).astype(np.float32)

    label_resampled = resample_volume(
        label,
        label_spacing,
        TARGET_SPACING,
        order=0,
    ).astype(np.int16)

    label_resampled = np.rint(label_resampled).astype(np.int16)

    if ct_resampled.shape != label_resampled.shape:
        raise ValueError(
            f"Resampled CT/label mismatch: "
            f"{ct_resampled.shape} vs {label_resampled.shape}"
        )

    return {
        "ct": ct_resampled,
        "label": label_resampled,
        "original_spacing": ct_spacing.tolist(),
        "target_spacing": TARGET_SPACING.tolist(),
    }


print("Preprocessing functions defined.")


Preprocessing functions defined.


##  18 — Remote source ZIP index cache


In [90]:
# Cell 18 — Remote source ZIP index cache

ZIP_INDEX_CACHE = {}

def get_source_zip_index(source_name):
    if source_name in ZIP_INDEX_CACHE:
        return ZIP_INDEX_CACHE[source_name]

    cfg = SOURCES[source_name]

    print(f"Inspecting remote image ZIP: {cfg['image_zip']}")
    size, entries = read_zip64_central_directory(cfg["image_url"])

    ZIP_INDEX_CACHE[source_name] = {
        "size": size,
        "entries": entries,
    }

    print(
        f"Remote ZIP indexed: {len(entries)} members, "
        f"{size / (1024**3):.2f} GB"
    )

    return ZIP_INDEX_CACHE[source_name]


print("ZIP index cache ready.")


ZIP index cache ready.


## 19 — Robust patient NIfTI loader


In [91]:
# Cell 19 — Robust patient NIfTI loader

def load_patient_nifti_from_source(patient_id, source_name):
    cfg = SOURCES[source_name]
    index = get_source_zip_index(source_name)
    entries = index["entries"]

    image_member = find_patient_member(entries, patient_id, "image")
    print("Image member:", image_member)
    image_bytes = get_zip_member_bytes(cfg["image_url"], entries[image_member])

    label_zip_path = CACHE_ROOT / cfg["label_zip"]
    cache_small_file(cfg["label_url"], label_zip_path)

    with zipfile.ZipFile(label_zip_path, "r") as zf:
        label_candidates = [
            name for name in zf.namelist()
            if patient_id in Path(name).name
            and "label" in Path(name).name.lower()
        ]
        if not label_candidates:
            raise FileNotFoundError(
                f"No label member found for {patient_id} in {cfg['label_zip']}"
            )
        label_member = sorted(label_candidates)[0]
        label_bytes = zf.read(label_member)

    print("Label member:", label_member)

    # NiBabel is given real temporary .nii.gz paths; BytesIO is not used.
    image_path = None
    label_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
            f.write(image_bytes)
            image_path = f.name
        with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
            f.write(label_bytes)
            label_path = f.name

        ct_nii = nib.load(image_path)
        label_nii = nib.load(label_path)

        # Materialize arrays while temporary files exist.
        ct_data = ct_nii.get_fdata(dtype=np.float32)
        label_data = label_nii.get_fdata(dtype=np.float32)
        ct_affine = ct_nii.affine.copy()
        label_affine = label_nii.affine.copy()
        ct_header = ct_nii.header.copy()
        label_header = label_nii.header.copy()
    finally:
        if image_path and os.path.exists(image_path):
            os.remove(image_path)
        if label_path and os.path.exists(label_path):
            os.remove(label_path)

    ct_nii = nib.Nifti1Image(ct_data, ct_affine, header=ct_header)
    label_nii = nib.Nifti1Image(label_data, label_affine, header=label_header)
    return ct_nii, label_nii

print("Robust patient loader defined.")


Robust patient loader defined.


## 20 — Persistent patient preprocessing and save helpers

In [92]:
# Cell 20 — Persistent patient preprocessing and save helpers

def patient_image_path(patient_id):
    return IMAGE_ROOT / f"{patient_id}-image.npy"


def patient_label_path(patient_id):
    return LABEL_ROOT / f"{patient_id}-label.npy"


def patient_is_complete(patient_id):
    return (
        patient_image_path(patient_id).exists()
        and patient_label_path(patient_id).exists()
        and patient_image_path(patient_id).stat().st_size > 0
        and patient_label_path(patient_id).stat().st_size > 0
    )


def update_status(status_df, status_path, patient_id, **updates):
    idx = status_df.index[status_df["public_id"] == str(patient_id)]
    if len(idx) != 1:
        raise ValueError(f"Status record missing for {patient_id}")
    for key, value in updates.items():
        if key not in status_df.columns:
            raise KeyError(key)
        status_df.loc[idx[0], key] = "" if value is None else str(value)
    status_df.to_csv(status_path, index=False)


def save_patient_arrays(patient_id, source_name, split_name, ct, label, original_spacing, status_df, status_path):
    ct_path = patient_image_path(patient_id)
    label_path = patient_label_path(patient_id)
    ct_tmp = IMAGE_ROOT / f".{patient_id}-image.tmp.npy"
    label_tmp = LABEL_ROOT / f".{patient_id}-label.tmp.npy"

    np.save(ct_tmp, ct)
    np.save(label_tmp, label)
    os.replace(ct_tmp, ct_path)
    os.replace(label_tmp, label_path)

    update_status(
        status_df, status_path, patient_id,
        status="completed",
        source=source_name,
        split=split_name,
        ct_path=str(ct_path),
        label_path=str(label_path),
        ct_shape=list(ct.shape),
        label_shape=list(label.shape),
        ct_dtype=str(ct.dtype),
        label_dtype=str(label.dtype),
        original_spacing=list(np.asarray(original_spacing, dtype=float)),
        target_spacing=list(TARGET_SPACING.astype(float)),
        ct_min=float(ct.min()),
        ct_max=float(ct.max()),
        label_ids=list(np.unique(label).astype(int)),
        ct_size_mb=ct_path.stat().st_size / (1024**2),
        label_size_mb=label_path.stat().st_size / (1024**2),
        completed_at=time.strftime("%Y-%m-%d %H:%M:%S"),
        error="",
    )
    return ct_path, label_path

print("Persistent save helpers defined.")


Persistent save helpers defined.


## Persistent processing status

`processing_status.csv` is the restart/audit manifest. Every patient has one record. A patient is marked `completed` only after both CT and label files have been written and the status file has been updated.


## 21 — Create or restore processing-status table

In [93]:
# Cell 21 — Create or restore processing-status table


STATUS_PATH = METADATA_ROOT / "processing_status.csv"
STATUS_COLUMNS = [
    "public_id", "source", "split", "status", "ct_path", "label_path",
    "ct_shape", "label_shape", "ct_dtype", "label_dtype", "original_spacing",
    "target_spacing", "ct_min", "ct_max", "label_ids", "ct_size_mb",
    "label_size_mb", "started_at", "completed_at", "error"
]

if STATUS_PATH.exists():
    status_df = pd.read_csv(STATUS_PATH, dtype=str)
else:
    status_df = pd.DataFrame(columns=STATUS_COLUMNS)

for col in STATUS_COLUMNS:
    if col not in status_df.columns:
        status_df[col] = ""
status_df = status_df[STATUS_COLUMNS]

registry_lookup = patient_registry.set_index("public_id")
existing = set(status_df["public_id"].astype(str))

for patient_id in patient_registry["public_id"].astype(str):
    if patient_id not in existing:
        row = registry_lookup.loc[patient_id]
        status_df.loc[len(status_df)] = [
            patient_id, row["source"], row["split"], "pending", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", ""
        ]

status_df.to_csv(STATUS_PATH, index=False)
print("Status records:", len(status_df))
print(status_df["status"].value_counts())
print("Saved:", STATUS_PATH)


Status records: 500
status
pending      499
completed      1
Name: count, dtype: int64
Saved: /kaggle/working/ribfrac_preprocessed_final/metadata/processing_status.csv


The output will help distinguish between:
- HTTP 206 or 200: the request is reachable; we should investigate why the ZIP index request is being rejected.
- HTTP 403: access is currently restricted for that URL or network.
- HTTP 429: requests are being rate-limited.
- Connection error: the request is failing before a normal HTTP response is received.
If the result is 403 again, we should avoid repeatedly requesting the same archive. We can then check the public Zenodo record and its alternate download route, while preserving your current preprocessing code and outputs.

## 22 — Smoke-test configuration

In [94]:

# ============================================================
# CELL #22 — SMOKE-TEST CONFIGURATION
# ============================================================

RUN_SMOKE_TEST = True

SMOKE_TEST_PATIENT = "RibFrac128"

SMOKE_TEST_SOURCE = "train_part1"


## 23 — Run and persist the smoke test


In [95]:

# CELL 23A — CHECK ZENODO DOWNLOAD ACCESS

import requests

url = SOURCES["train_part1"]["image_url"] + "?download=1"

response = None

try:
    response = requests.get(
        url,
        headers={"Range": "bytes=0-0"},
        stream=True,
        timeout=30,
        allow_redirects=True
    )

    print("HTTP status:", response.status_code)
    print("Content-Range:", response.headers.get("Content-Range"))
    print("Accept-Ranges:", response.headers.get("Accept-Ranges"))

    if response.status_code == 206:
        print("PASS: Byte-range download is supported.")
        SOURCES["train_part1"]["image_url"] = url

    elif response.status_code == 403:
        print("BLOCKED: Zenodo denied the request.")

    else:
        print("Unexpected response; do not run the ZIP reader yet.")

except requests.RequestException as exc:
    print("Connection error:", exc)

finally:
    if response is not None:
        response.close()


HTTP status: 206
Content-Range: bytes 0-0/36605667168
Accept-Ranges: bytes
PASS: Byte-range download is supported.


In [96]:
# ============================================================
# CELL 23B — FIX ZENODO HTTP 403 USING REQUESTS
# ============================================================

import requests
import re
import time

# Ensure the tested download URL is retained
for key in ("image_url", "label_url"):
    url = SOURCES["train_part1"][key]
    if "download=1" not in url:
        separator = "&" if "?" in url else "?"
        SOURCES["train_part1"][key] = (
            url + separator + "download=1"
        )


def get_remote_size(url, timeout=120):
    """Obtain archive size without downloading the ZIP."""

    with requests.get(
        url,
        headers={"Range": "bytes=0-0"},
        stream=True,
        timeout=(30, timeout),
        allow_redirects=True
    ) as response:

        response.raise_for_status()

        if response.status_code != 206:
            raise RuntimeError(
                f"Expected HTTP 206, got {response.status_code}"
            )

        content_range = response.headers.get(
            "Content-Range", ""
        )

        match = re.fullmatch(
            r"bytes\s+0-0/(\d+)",
            content_range.strip()
        )

        if not match:
            raise RuntimeError(
                f"Invalid Content-Range: {content_range}"
            )

        return int(match.group(1))


def http_get_bytes(
    url,
    start=None,
    end=None,
    timeout=180,
    max_retries=6
):
    """Retrieve exact byte ranges using requests."""

    if (start is None) != (end is None):
        raise ValueError("Specify both start and end.")

    ranged = start is not None

    headers = {}

    if ranged:
        if start < 0 or end < start:
            raise ValueError("Invalid byte range.")
        headers["Range"] = f"bytes={start}-{end}"

    for attempt in range(max_retries):

        try:
            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, timeout),
                allow_redirects=True
            ) as response:

                response.raise_for_status()

                if ranged:
                    if response.status_code != 206:
                        raise RuntimeError(
                            "Server ignored byte-range request. "
                            "Full archive download prevented."
                        )

                    expected_range = (
                        f"bytes {start}-{end}/"
                    )

                    actual_range = response.headers.get(
                        "Content-Range", ""
                    )

                    if not actual_range.startswith(expected_range):
                        raise RuntimeError(
                            f"Unexpected Content-Range: {actual_range}"
                        )

                limit = (
                    end - start + 1
                    if ranged
                    else 1024**3
                )

                chunks = []
                received = 0

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if not chunk:
                        continue

                    received += len(chunk)

                    if received > limit:
                        raise RuntimeError(
                            "Download exceeded permitted size."
                        )

                    chunks.append(chunk)

                data = b"".join(chunks)

                if ranged and len(data) != end - start + 1:
                    raise RuntimeError(
                        f"Incomplete download: {len(data)} bytes"
                    )

                return data, dict(response.headers)

        except requests.exceptions.RequestException as exc:

            status = getattr(
                getattr(exc, "response", None),
                "status_code",
                None
            )

            # Do not repeatedly retry forbidden requests
            if (
                status is not None
                and 400 <= status < 500
                and status not in (408, 429)
            ):
                raise

            if attempt == max_retries - 1:
                raise

            time.sleep(min(15, 2 ** attempt))


ZIP_INDEX_CACHE.pop("train_part1", None)

print("Requests-based ZIP download helpers: READY")


Requests-based ZIP download helpers: READY


In [97]:
# ============================================================
# CELL #23 — RUN AND PERSIST THE SMOKE TEST
# RibFrac 500-Patient Preprocessing
# ============================================================

import gc
import numpy as np

if RUN_SMOKE_TEST:

    patient_id = SMOKE_TEST_PATIENT
    source_name = SMOKE_TEST_SOURCE

    # Verify patient against authoritative registry
    patient_row = patient_registry.loc[
        patient_registry["public_id"].astype(str) == patient_id
    ]

    assert len(patient_row) == 1, (
        f"Patient {patient_id} missing from registry."
    )

    assert patient_row.iloc[0]["source"] == source_name

    split_name = str(patient_row.iloc[0]["split"])

    print("=" * 70)
    print("RIBFRAC SMOKE TEST")
    print("=" * 70)

    print("Patient:", patient_id)
    print("Source:", source_name)
    print("Split:", split_name)

    # --------------------------------------------------------
    # 1. Reuse existing processed files where available
    # --------------------------------------------------------

    already_saved = patient_is_complete(patient_id)

    if already_saved:

        print("\nExisting CT and label files found.")
        print("Loading saved arrays for validation.")

        ct = np.load(
            patient_image_path(patient_id),
            mmap_mode="r"
        )

        label = np.load(
            patient_label_path(patient_id),
            mmap_mode="r"
        )

    else:

        print("\nNo complete local pair found.")
        print("Loading source NIfTI volumes.")

        # Check remote ZIP dependency
        if not callable(globals().get("get_remote_size")):
            raise RuntimeError(
                "get_remote_size() is undefined. "
                "Execute Cell #5A before continuing."
            )

        ct_nii, label_nii = load_patient_nifti_from_source(
            patient_id,
            source_name
        )

        result = preprocess_nifti_pair(
            ct_nii,
            label_nii
        )

        ct = result["ct"]
        label = result["label"]

        original_spacing = result["original_spacing"]

    # --------------------------------------------------------
    # 2. Validate the preprocessed arrays
    # --------------------------------------------------------

    assert ct.ndim == 3, "CT must be 3D."
    assert label.ndim == 3, "Label must be 3D."

    assert ct.shape == label.shape, (
        "CT and label shapes do not match."
    )

    assert ct.dtype == np.float32
    assert label.dtype == np.int16

    assert np.isfinite(ct).all(), (
        "Non-finite values found in CT."
    )

    assert float(ct.min()) >= 0.0
    assert float(ct.max()) <= 1.0

    print("\nPREPROCESSING VALIDATION")
    print("-" * 70)

    print("CT shape:", ct.shape)
    print("Label shape:", label.shape)
    print("CT dtype:", ct.dtype)
    print("Label dtype:", label.dtype)
    print("CT range:", float(ct.min()), float(ct.max()))
    print("Label IDs:", np.unique(label))
    print("Target spacing:", TARGET_SPACING)

    print("\nSmoke-test preprocessing: PASS")

    # --------------------------------------------------------
    # 3. Persist newly preprocessed arrays
    # --------------------------------------------------------

    if not already_saved:

        ct_path, label_path = save_patient_arrays(
            patient_id,
            source_name,
            split_name,
            ct,
            label,
            original_spacing,
            status_df,
            STATUS_PATH
        )

        print("\nNewly preprocessed files saved.")

    else:

        ct_path = patient_image_path(patient_id)
        label_path = patient_label_path(patient_id)

        print("\nExisting files retained; no overwrite.")

    print("CT file:", ct_path)
    print("Label file:", label_path)

    print("\nSMOKE TEST: PASS")
    print("=" * 70)

    # Release memory
    del ct, label

    if not already_saved:
        del ct_nii, label_nii, result

    gc.collect()

else:
    print("Smoke test disabled.")


RIBFRAC SMOKE TEST
Patient: RibFrac128
Source: train_part1
Split: train

Existing CT and label files found.
Loading saved arrays for validation.

PREPROCESSING VALIDATION
----------------------------------------------------------------------
CT shape: (395, 395, 421)
Label shape: (395, 395, 421)
CT dtype: float32
Label dtype: int16
CT range: 0.0 1.0
Label IDs: [0 1 2 3 4 5 6 7 8]
Target spacing: [1. 1. 1.]

Smoke-test preprocessing: PASS

Existing files retained; no overwrite.
CT file: /kaggle/working/ribfrac_preprocessed_final/images/RibFrac128-image.npy
Label file: /kaggle/working/ribfrac_preprocessed_final/labels/RibFrac128-label.npy

SMOKE TEST: PASS


In [98]:

# CELL 23C — VERIFY REMOTE ZIP RANGE ACCESS

url = SOURCES["train_part1"]["image_url"]

size = get_remote_size(url)

print("Remote ZIP size:", size)

first_byte, _ = http_get_bytes(
    url, start=0, end=0, max_retries=1
)

last_byte, _ = http_get_bytes(
    url, start=size-1, end=size-1, max_retries=1
)

assert len(first_byte) == 1
assert len(last_byte) == 1

print("First-byte access: PASS")
print("Last-byte access: PASS")
print("REMOTE ZIP RANGE ACCESS: PASS")


Remote ZIP size: 36605667168
First-byte access: PASS
Last-byte access: PASS
REMOTE ZIP RANGE ACCESS: PASS


In [72]:
index = get_source_zip_index("train_part1")

print("ZIP INDEXING: PASS")
print("Members:", len(index["entries"]))

ZIP INDEXING: PASS
Members: 301


## Full 500-patient persistent preprocessing

The next cell is resumable and processes patients one at a time. Set `MAX_PATIENTS_PER_RUN` to a small number while testing. Because a 1-mm CT volume is large, do not assume that all 500 volumes will fit in the Kaggle working disk. If storage is insufficient, process in durable batches and retain the same patient manifest.


In [101]:

# ============================================================
# CELL 24A — FIX MISSING IMPORT
# ============================================================

import shutil
import gc
import time
import numpy as np
import pandas as pd

# Verify disk space
WORK_ROOT = METADATA_ROOT.parent

free_gb = shutil.disk_usage(WORK_ROOT).free / (1024**3)

print("Import verification: PASS")
print(f"Available disk space: {free_gb:.2f} GiB")
print("Working directory:", WORK_ROOT)


Import verification: PASS
Available disk space: 19.13 GiB
Working directory: /kaggle/working/ribfrac_preprocessed_final


In [111]:

# ============================================================
# CHECK BOTH PERSISTED AND WORKING DATASETS
# ============================================================

from pathlib import Path

OLD_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

NEW_ROOT = Path(
    "/kaggle/working/ribfrac_preprocessed_final"
)

def find_existing_patient_pair(patient_id):

    patient_id = str(patient_id)

    # Prefer the current working copy
    for root in [NEW_ROOT, OLD_ROOT]:

        image_path = (
            root / "images" / f"{patient_id}-image.npy"
        )

        label_path = (
            root / "labels" / f"{patient_id}-label.npy"
        )

        if (
            image_path.is_file()
            and label_path.is_file()
            and image_path.stat().st_size > 0
            and label_path.stat().st_size > 0
        ):
            return image_path, label_path

    return None


print("Existing-patient lookup helper: READY")


Existing-patient lookup helper: READY


In [109]:

# ============================================================
# CELL #24 — RESUMABLE FULL PREPROCESSING LOOP
# RibFrac 500-Patient Dataset
# ============================================================

# ------------------------------------------------------------
# 1. Processing configuration
# ------------------------------------------------------------

MAX_PATIENTS_PER_RUN = 5
RESUME_EXISTING = True
RETRY_FAILED = True

# Verify successful smoke test
assert patient_is_complete("RibFrac128"), (
    "Cell #23 must successfully save RibFrac128 "
    "before full preprocessing begins."
)

# Restore current processing status
status_df = pd.read_csv(
    STATUS_PATH,
    dtype=str,
    keep_default_na=False
)

assert len(status_df) == 500
assert status_df["public_id"].nunique() == 500
assert patient_registry["public_id"].nunique() == 500

# ------------------------------------------------------------
# 2. Establish processing order
# ------------------------------------------------------------

split_priority = {
    "train": 0,
    "validation": 1,
    "test": 2
}

registry_to_process = patient_registry.copy()

registry_to_process["_priority"] = (
    registry_to_process["split"].map(split_priority)
)

assert registry_to_process["_priority"].notna().all()

registry_to_process = (
    registry_to_process
    .sort_values(["_priority", "public_id"])
    .drop(columns="_priority")
    .reset_index(drop=True)
)

print("=" * 70)
print("RIBFRAC 500-PATIENT PREPROCESSING")
print("=" * 70)

print("\nPatient distribution:")
print(registry_to_process["split"].value_counts())

print("\nMaximum attempts this run:", MAX_PATIENTS_PER_RUN)
print("Resume existing:", RESUME_EXISTING)
print("Retry failed:", RETRY_FAILED)

# ------------------------------------------------------------
# 3. Initialize counters
# ------------------------------------------------------------

attempted_this_run = 0
processed_this_run = 0
failed_this_run = 0
skipped_this_run = 0

# ------------------------------------------------------------
# 4. Process patients sequentially
# ------------------------------------------------------------

for _, row in registry_to_process.iterrows():

    patient_id = str(row["public_id"])
    source_name = str(row["source"])
    split_name = str(row["split"])

    current_status = status_df.loc[
        status_df["public_id"] == patient_id,
        "status"
    ].iloc[0]

    # --------------------------------------------------------
    # Skip completed patients
    # --------------------------------------------------------
     
# Check both old and new datasets
existing_pair = find_existing_patient_pair(patient_id)

if RESUME_EXISTING and existing_pair is not None:

    existing_ct, existing_label = existing_pair

    update_status(
        status_df,
        STATUS_PATH,
        patient_id,
        status="completed",
        source=source_name,
        split=split_name,
        ct_path=str(existing_ct),
        label_path=str(existing_label),
        error=""
    )

    skipped_this_run += 1

    print(
        f"{patient_id}: already available — SKIP"
    )

    continue
 
    

    # Skip failed patients when retry disabled
    if current_status == "failed" and not RETRY_FAILED:
        continue

    # Stop after 5 attempted patients
    if attempted_this_run >= MAX_PATIENTS_PER_RUN:
        print("\nBatch limit reached.")
        break

    # Check remaining disk space
    free_gb = shutil.disk_usage(WORK_ROOT).free / (1024**3)

    if free_gb < 2.0:
        print(
            f"\nInsufficient free storage: {free_gb:.2f} GiB. "
            "Stopping before the next patient."
        )
        break

    attempted_this_run += 1

    print("\n" + "=" * 70)
    print(
        f"PROCESSING PATIENT "
        f"{attempted_this_run}/{MAX_PATIENTS_PER_RUN}"
    )
    print("=" * 70)

    print("Patient:", patient_id)
    print("Source:", source_name)
    print("Split:", split_name)
    print(f"Available disk space: {free_gb:.2f} GiB")

    # Record processing start
    update_status(
        status_df,
        STATUS_PATH,
        patient_id,
        status="processing",
        source=source_name,
        split=split_name,
        started_at=time.strftime("%Y-%m-%d %H:%M:%S"),
        error=""
    )

    ct_nii = None
    label_nii = None
    preprocessed = None
    ct = None
    label = None

    try:

        # ----------------------------------------------------
        # A. Load original CT and fracture annotation
        # ----------------------------------------------------

        ct_nii, label_nii = load_patient_nifti_from_source(
            patient_id,
            source_name
        )

        # ----------------------------------------------------
        # B. Apply established preprocessing
        # ----------------------------------------------------

        preprocessed = preprocess_nifti_pair(
            ct_nii,
            label_nii
        )

        ct = preprocessed["ct"]
        label = preprocessed["label"]

        original_spacing = preprocessed["original_spacing"]

        # ----------------------------------------------------
        # C. Validate preprocessed arrays
        # ----------------------------------------------------

        assert ct.ndim == 3
        assert label.ndim == 3
        assert ct.shape == label.shape

        assert ct.dtype == np.float32
        assert label.dtype == np.int16

        assert np.isfinite(ct).all()
        assert 0.0 <= float(ct.min()) <= float(ct.max()) <= 1.0

        # ----------------------------------------------------
        # D. Save CT, label, and processing metadata
        # ----------------------------------------------------

        ct_path, label_path = save_patient_arrays(
            patient_id,
            source_name,
            split_name,
            ct,
            label,
            original_spacing,
            status_df,
            STATUS_PATH
        )

        assert ct_path.is_file()
        assert label_path.is_file()

        processed_this_run += 1

        print("\nSTATUS: COMPLETED")
        print("CT shape:", ct.shape)
        print("Label shape:", label.shape)
        print("CT saved:", ct_path)
        print("Label saved:", label_path)

    except Exception as exc:

        failed_this_run += 1
        error_text = repr(exc)

        print("\nSTATUS: FAILED")
        print("Error:", error_text)

        update_status(
            status_df,
            STATUS_PATH,
            patient_id,
            status="failed",
            error=error_text,
            completed_at=""
        )

        # Stop on remote HTTP errors
        if any(
            term in error_text.lower()
            for term in [
                "403", "429", "forbidden",
                "httperror", "rate limit"
            ]
        ):
            print("Remote HTTP error detected. Stopping batch.")
            break

    finally:

        # Release memory after each patient
        del ct_nii, label_nii, preprocessed, ct, label

        gc.collect()

# ------------------------------------------------------------
# 5. Reload saved status table
# ------------------------------------------------------------

status_df = pd.read_csv(
    STATUS_PATH,
    dtype=str,
    keep_default_na=False
)

# ------------------------------------------------------------
# 6. Batch summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RIBFRAC PREPROCESSING BATCH SUMMARY")
print("=" * 70)

print("Attempted this run:", attempted_this_run)
print("Completed this run:", processed_this_run)
print("Failed this run:", failed_this_run)
print("Existing patients skipped:", skipped_this_run)

print("\nCurrent patient processing status:")
print(status_df["status"].value_counts())

print("\nProcessing status saved to:")
print(STATUS_PATH)

print("=" * 70)


RIBFRAC 500-PATIENT PREPROCESSING

Patient distribution:
split
train         350
validation     75
test           75
Name: count, dtype: int64

Maximum attempts this run: 5
Resume existing: True
Retry failed: True

PROCESSING PATIENT 1/5
Patient: RibFrac143
Source: train_part1
Split: train
Available disk space: 8.22 GiB
Image member: Part1/RibFrac143-image.nii.gz
Label member: Part1/RibFrac143-label.nii.gz

STATUS: COMPLETED
CT shape: (493, 493, 456)
Label shape: (493, 493, 456)
CT saved: /kaggle/working/ribfrac_preprocessed_final/images/RibFrac143-image.npy
Label saved: /kaggle/working/ribfrac_preprocessed_final/labels/RibFrac143-label.npy

PROCESSING PATIENT 2/5
Patient: RibFrac144
Source: train_part1
Split: train
Available disk space: 7.60 GiB
Image member: Part1/RibFrac144-image.nii.gz
Label member: Part1/RibFrac144-label.nii.gz

STATUS: COMPLETED
CT shape: (414, 414, 446)
Label shape: (414, 414, 446)
CT saved: /kaggle/working/ribfrac_preprocessed_final/images/RibFrac144-image.npy


In [112]:

# ============================================================
# CELL #24C — PREPARE PERSISTENCE CHECKPOINT
# ============================================================

from pathlib import Path
import os
import shutil
import numpy as np
import pandas as pd

OLD_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

WORK_ROOT = Path(
    "/kaggle/working/ribfrac_preprocessed_final"
)

META_ROOT = WORK_ROOT / "metadata"
STATUS_PATH = META_ROOT / "processing_status.csv"

for name in ["images", "labels", "metadata"]:
    assert (OLD_ROOT / name).is_dir(), (
        f"Missing original directory: {name}"
    )
    assert (WORK_ROOT / name).is_dir(), (
        f"Missing working directory: {name}"
    )

# ------------------------------------------------------------
# 1. Confirm authoritative patient assignments
# ------------------------------------------------------------

def read_assignments(path):
    df = pd.read_csv(path, dtype=str)

    if "public_id" not in df.columns:
        df = df.rename(
            columns={"patient_id": "public_id"}
        )

    df = df[["public_id", "split"]].copy()
    df["split"] = df["split"].str.strip().str.lower()
    df["split"] = df["split"].replace({"val": "validation"})

    assert len(df) == 500
    assert df["public_id"].nunique() == 500

    return df.sort_values("public_id").reset_index(drop=True)


old_split = read_assignments(
    OLD_ROOT / "metadata/patient_split_500.csv"
)

work_split = read_assignments(
    META_ROOT / "patient_split_500.csv"
)

assert old_split.equals(work_split), (
    "Patient assignments differ. Persistence stopped."
)

valid_ids = set(old_split["public_id"])

# ------------------------------------------------------------
# 2. Locate complete CT-label pairs
# ------------------------------------------------------------

def complete_pairs(root):

    images = {
        p.name.removesuffix("-image.npy")
        for p in (root / "images").glob("*-image.npy")
    }

    labels = {
        p.name.removesuffix("-label.npy")
        for p in (root / "labels").glob("*-label.npy")
    }

    assert images == labels, (
        f"Incomplete CT-label pairs in {root}"
    )

    return images


old_ids = complete_pairs(OLD_ROOT)
work_ids = complete_pairs(WORK_ROOT)

all_ids = old_ids | work_ids

assert all_ids.issubset(valid_ids)

print("Previously persisted:", len(old_ids))
print("Working patients:", len(work_ids))
print("Total unique patients:", len(all_ids))

# ------------------------------------------------------------
# 3. Check available storage
# ------------------------------------------------------------

missing_ids = old_ids - work_ids

bytes_required = sum(
    (
        OLD_ROOT / folder / f"{pid}-{suffix}.npy"
    ).stat().st_size
    for pid in missing_ids
    for folder, suffix in [
        ("images", "image"),
        ("labels", "label")
    ]
)

free_bytes = shutil.disk_usage(WORK_ROOT).free
reserve_bytes = 1024**3  # 1 GiB reserve

print(
    "Storage needed for merge:",
    round(bytes_required / 1024**3, 2),
    "GiB"
)
print(
    "Free working storage:",
    round(free_bytes / 1024**3, 2),
    "GiB"
)

if free_bytes < bytes_required + reserve_bytes:
    raise RuntimeError(
        "Insufficient disk space to merge the original "
        "patient pairs safely. No files have been copied."
    )

# ------------------------------------------------------------
# 4. Copy missing original patients
# ------------------------------------------------------------

for pid in sorted(missing_ids):

    for folder, suffix in [
        ("images", "image"),
        ("labels", "label")
    ]:

        filename = f"{pid}-{suffix}.npy"

        src = OLD_ROOT / folder / filename
        dst = WORK_ROOT / folder / filename

        tmp = dst.with_name(dst.name + ".copying")

        shutil.copy2(src, tmp)

        assert tmp.stat().st_size == src.stat().st_size

        os.replace(tmp, dst)

    print("Restored:", pid)

# ------------------------------------------------------------
# 5. Validate complete pairs without loading full volumes
# ------------------------------------------------------------

verified = []

registry = patient_registry.copy()
registry["public_id"] = registry["public_id"].astype(str)
registry = registry.set_index("public_id")

for pid in sorted(all_ids):

    ct_path = WORK_ROOT / "images" / f"{pid}-image.npy"
    label_path = WORK_ROOT / "labels" / f"{pid}-label.npy"

    ct = np.load(ct_path, mmap_mode="r", allow_pickle=False)
    label = np.load(label_path, mmap_mode="r", allow_pickle=False)

    assert ct.ndim == 3
    assert label.ndim == 3
    assert ct.shape == label.shape
    assert ct.dtype == np.float32
    assert label.dtype == np.int16

    verified.append({
        "public_id": pid,
        "source": registry.loc[pid, "source"],
        "split": registry.loc[pid, "split"],
        "ct_shape": str(tuple(ct.shape)),
        "label_shape": str(tuple(label.shape)),
        "ct_dtype": str(ct.dtype),
        "label_dtype": str(label.dtype),
        "ct_size_mb": round(
            ct_path.stat().st_size / 1024**2, 3
        ),
        "label_size_mb": round(
            label_path.stat().st_size / 1024**2, 3
        ),
    })

    del ct, label

inventory = pd.DataFrame(verified)

# ------------------------------------------------------------
# 6. Update processing-status registry
# ------------------------------------------------------------

status = pd.read_csv(
    STATUS_PATH,
    dtype=str,
    keep_default_na=False
)

assert len(status) == 500
assert status["public_id"].nunique() == 500
assert set(status["public_id"]) == valid_ids

# Keep a backup before modifying the status file
backup_path = META_ROOT / "processing_status_before_merge.csv"

if not backup_path.exists():
    shutil.copy2(STATUS_PATH, backup_path)

status = status.set_index("public_id")

for item in verified:

    pid = item["public_id"]

    status.loc[pid, "status"] = "completed"
    status.loc[pid, "source"] = item["source"]
    status.loc[pid, "split"] = item["split"]

    # Portable paths for the published dataset
    status.loc[pid, "ct_path"] = f"images/{pid}-image.npy"
    status.loc[pid, "label_path"] = f"labels/{pid}-label.npy"

    for col in [
        "ct_shape", "label_shape",
        "ct_dtype", "label_dtype",
        "ct_size_mb", "label_size_mb"
    ]:
        status.loc[pid, col] = str(item[col])

    status.loc[pid, "error"] = ""

status = status.reset_index()

# Save checkpoint metadata
inventory_path = META_ROOT / "persisted_patient_inventory.csv"
inventory.to_csv(inventory_path, index=False)

tmp_status = META_ROOT / "processing_status.tmp.csv"
status.to_csv(tmp_status, index=False)
os.replace(tmp_status, STATUS_PATH)

# ------------------------------------------------------------
# 7. Final persistence preparation report
# ------------------------------------------------------------

assert complete_pairs(WORK_ROOT) == all_ids

print("\n" + "=" * 60)
print("PERSISTENCE CHECKPOINT VALIDATION")
print("=" * 60)

print("Total verified patient pairs:", len(inventory))
print("CT files:", len(list((WORK_ROOT / "images").glob("*-image.npy"))))
print("Label files:", len(list((WORK_ROOT / "labels").glob("*-label.npy"))))

print("\nVerified patients by split:")
print(inventory["split"].value_counts())

print("\nStatus registry:")
print(status["status"].value_counts())

print("\nCHECKPOINT PREPARATION: PASS")
print("=" * 60)


Previously persisted: 13
Working patients: 36
Total unique patients: 49
Storage needed for merge: 4.66 GiB
Free working storage: 6.15 GiB
Restored: RibFrac107
Restored: RibFrac109
Restored: RibFrac114
Restored: RibFrac12
Restored: RibFrac120
Restored: RibFrac121
Restored: RibFrac129
Restored: RibFrac134
Restored: RibFrac140
Restored: RibFrac149
Restored: RibFrac15
Restored: RibFrac156
Restored: RibFrac167

PERSISTENCE CHECKPOINT VALIDATION
Total verified patient pairs: 49
CT files: 49
Label files: 49

Verified patients by split:
split
train    36
test     13
Name: count, dtype: int64

Status registry:
status
pending      451
completed     49
Name: count, dtype: int64

CHECKPOINT PREPARATION: PASS


In [110]:

# ============================================================
# CHECK OLD AND NEW PREPROCESSED PATIENTS
# ============================================================

from pathlib import Path

OLD_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

NEW_ROOT = Path(
    "/kaggle/working/ribfrac_preprocessed_final"
)

def get_complete_patients(root):

    images = {
        p.name.removesuffix("-image.npy")
        for p in (root / "images").glob("*-image.npy")
    }

    labels = {
        p.name.removesuffix("-label.npy")
        for p in (root / "labels").glob("*-label.npy")
    }

    return images & labels


old_patients = get_complete_patients(OLD_ROOT)
new_patients = get_complete_patients(NEW_ROOT)

combined_patients = old_patients | new_patients

print("=" * 60)
print("RIBFRAC DATASET PERSISTENCE AUDIT")
print("=" * 60)

print("Previously persisted:", len(old_patients))
print("Current working patients:", len(new_patients))
print("Patients in both locations:",
      len(old_patients & new_patients))
print("Total unique completed patients:",
      len(combined_patients))

print("Remaining patients:", 500 - len(combined_patients))

print("=" * 60)


RIBFRAC DATASET PERSISTENCE AUDIT
Previously persisted: 13
Current working patients: 36
Patients in both locations: 0
Total unique completed patients: 49
Remaining patients: 451


In [113]:

# ============================================================
# CELL #24B — PUBLISH RIBFRAC CHECKPOINT TO KAGGLE
# ============================================================

import kagglehub
from pathlib import Path

DATASET_HANDLE = "waboke/ribfrac-preprocessed-final"

PUBLISH_ROOT = Path(
    "/kaggle/working/ribfrac_preprocessed_final"
)

# Final verification before upload
image_files = list(
    (PUBLISH_ROOT / "images").glob("*-image.npy")
)

label_files = list(
    (PUBLISH_ROOT / "labels").glob("*-label.npy")
)

assert len(image_files) == 49
assert len(label_files) == 49

assert (
    PUBLISH_ROOT / "metadata/patient_split_500.csv"
).is_file()

assert (
    PUBLISH_ROOT / "metadata/processing_status.csv"
).is_file()

print("=" * 60)
print("PUBLISHING RIBFRAC DATASET CHECKPOINT")
print("=" * 60)

print("Dataset:", DATASET_HANDLE)
print("CT volumes:", len(image_files))
print("Fracture masks:", len(label_files))

# Publish a new version of the existing dataset
kagglehub.dataset_upload(
    DATASET_HANDLE,
    str(PUBLISH_ROOT),
    version_notes=(
        "RibFrac preprocessing checkpoint: "
        "49 verified CT-label pairs "
        "(13 previous + 36 additional). "
        "Original 500-patient 350/75/75 split preserved."
    ),
    ignore_patterns=[
        "cache/",
        "smoke_test/",
        "*.copying",
        "*.partial",
        "*.tmp.npy"
    ]
)

print("\nUpload request completed.")
print(
    "Check the Kaggle dataset Versions page "
    "to confirm publication."
)


PUBLISHING RIBFRAC DATASET CHECKPOINT
Dataset: waboke/ribfrac-preprocessed-final
CT volumes: 49
Fracture masks: 49
Uploading Dataset https://kaggle.com/datasets/waboke/ribfrac-preprocessed-final ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpdf18rgrl/archive.zip

Uploading: 100%|██████████| 19.3G/19.3G [05:03<00:00, 63.8MB/s]

Upload successful: /tmp/tmpdf18rgrl/archive.zip (18GB)


Your dataset version has been created.
Files are being processed...
See at: https://kaggle.com/datasets/waboke/ribfrac-preprocessed-final

Upload request completed.
Check the Kaggle dataset Versions page to confirm publication.


In [114]:

# ============================================================
# CELL #25 — PERSISTENT OUTPUT INTEGRITY VALIDATION
# RibFrac 500-Patient Preprocessing
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Configure checkpoint locations
# ------------------------------------------------------------

CHECKPOINT_ROOT = Path(
    "/kaggle/working/ribfrac_preprocessed_final"
)

IMAGE_DIR = CHECKPOINT_ROOT / "images"
LABEL_DIR = CHECKPOINT_ROOT / "labels"

STATUS_PATH = (
    CHECKPOINT_ROOT / "metadata/processing_status.csv"
)

EXPECTED_COMPLETED = 49
EXPECTED_TOTAL = 500

# ------------------------------------------------------------
# 2. Load processing status
# ------------------------------------------------------------

status_df = pd.read_csv(
    STATUS_PATH,
    dtype=str,
    keep_default_na=False
)

assert len(status_df) == EXPECTED_TOTAL
assert status_df["public_id"].nunique() == EXPECTED_TOTAL

completed = status_df[
    status_df["status"] == "completed"
].copy()

completed_ids = set(completed["public_id"])

print("=" * 65)
print("RIBFRAC — PERSISTENT OUTPUT INTEGRITY VALIDATION")
print("=" * 65)

print("\nProcessing status:")
print(status_df["status"].value_counts())

# ------------------------------------------------------------
# 3. Identify saved CT and label arrays
# ------------------------------------------------------------

image_ids = {
    p.name.removesuffix("-image.npy")
    for p in IMAGE_DIR.glob("*-image.npy")
}

label_ids = {
    p.name.removesuffix("-label.npy")
    for p in LABEL_DIR.glob("*-label.npy")
}

complete_ids = image_ids & label_ids

print("\nSaved files:")
print("CT images:", len(image_ids))
print("Fracture labels:", len(label_ids))
print("Complete pairs:", len(complete_ids))

# ------------------------------------------------------------
# 4. Verify file completeness and status consistency
# ------------------------------------------------------------

assert image_ids == label_ids, (
    "Some CT images or fracture labels are missing."
)

assert completed_ids == complete_ids, (
    "Completed status records do not match saved pairs."
)

assert len(completed_ids) == EXPECTED_COMPLETED, (
    f"Expected {EXPECTED_COMPLETED} completed patients, "
    f"found {len(completed_ids)}."
)

# ------------------------------------------------------------
# 5. Verify patient assignments
# ------------------------------------------------------------

registry_lookup = (
    patient_registry
    .set_index("public_id")
)

assert complete_ids.issubset(
    set(registry_lookup.index)
)

for _, row in completed.iterrows():

    patient_id = row["public_id"]

    assert row["split"] == (
        registry_lookup.loc[patient_id, "split"]
    )

    assert row["source"] == (
        registry_lookup.loc[patient_id, "source"]
    )

print("\nPatient assignment consistency: PASS")

# ------------------------------------------------------------
# 6. Verify CT and label array integrity
# ------------------------------------------------------------

validation_errors = []
validated = 0

for patient_id in sorted(completed_ids):

    ct_path = (
        IMAGE_DIR / f"{patient_id}-image.npy"
    )

    label_path = (
        LABEL_DIR / f"{patient_id}-label.npy"
    )

    try:

        ct = np.load(
            ct_path,
            mmap_mode="r",
            allow_pickle=False
        )

        label = np.load(
            label_path,
            mmap_mode="r",
            allow_pickle=False
        )

        assert ct.ndim == 3
        assert label.ndim == 3

        assert ct.shape == label.shape

        assert ct.dtype == np.float32
        assert label.dtype == np.int16

        validated += 1

        del ct, label

    except Exception as exc:

        validation_errors.append(
            (patient_id, repr(exc))
        )

# ------------------------------------------------------------
# 7. Final report
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRITY VALIDATION RESULTS")
print("-" * 65)

print("Completed patients:", len(completed_ids))
print("Validated CT-label pairs:", validated)
print("Validation errors:", len(validation_errors))

print("\nCompleted patients by split:")
print(completed["split"].value_counts())

if validation_errors:
    print("\nErrors detected:")
    for patient_id, error in validation_errors:
        print(patient_id, error)

assert not validation_errors, (
    "One or more patient pairs failed validation."
)

print("\nPERSISTENT OUTPUT INTEGRITY: PASS")
print("=" * 65)


RIBFRAC — PERSISTENT OUTPUT INTEGRITY VALIDATION

Processing status:
status
pending      451
completed     49
Name: count, dtype: int64

Saved files:
CT images: 49
Fracture labels: 49
Complete pairs: 49

Patient assignment consistency: PASS

-----------------------------------------------------------------
INTEGRITY VALIDATION RESULTS
-----------------------------------------------------------------
Completed patients: 49
Validated CT-label pairs: 49
Validation errors: 0

Completed patients by split:
split
train    36
test     13
Name: count, dtype: int64

PERSISTENT OUTPUT INTEGRITY: PASS


In [115]:

# Cell 26 — Validate one persisted patient

completed_ids = completed["public_id"].tolist()

if completed_ids:
    patient_id = completed_ids[0]

    ct = np.load(
        patient_image_path(patient_id),
        mmap_mode="r"
    )

    label = np.load(
        patient_label_path(patient_id),
        mmap_mode="r"
    )

    print("Patient:", patient_id)
    print("CT shape:", ct.shape)
    print("Label shape:", label.shape)
    print("CT dtype:", ct.dtype)
    print("Label dtype:", label.dtype)
    print("CT range:", float(ct.min()), "to", float(ct.max()))
    print("Label IDs:", np.unique(label))

    assert ct.shape == label.shape
    assert ct.dtype == np.float32
    assert label.dtype == np.int16
    assert float(ct.min()) >= 0.0 and float(ct.max()) <= 1.0

    print("Persisted-volume validation: PASS")

else:
    print("No completed patients yet.")


Patient: RibFrac10
CT shape: (381, 381, 451)
Label shape: (381, 381, 451)
CT dtype: float32
Label dtype: int16
CT range: 0.0 to 1.0
Label IDs: [0 1 2 3 4 5 6 7 8]
Persisted-volume validation: PASS


In [116]:

# ============================================================
# CELL #27 — FINAL PROCESSING REPORT AND OUTPUT INVENTORY
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

WORK_ROOT = Path(
    "/kaggle/working/ribfrac_preprocessed_final"
)

STATUS_PATH = (
    WORK_ROOT / "metadata/processing_status.csv"
)

status_df = pd.read_csv(
    STATUS_PATH,
    dtype=str,
    keep_default_na=False
)

print("=" * 80)
print("NOTEBOOK 02 — PERSISTENT PROCESSING REPORT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Overall processing status
# ------------------------------------------------------------

print("\nPROCESSING STATUS")
print("-" * 60)

print(status_df["status"].value_counts())

assert len(status_df) == 500
assert status_df["public_id"].nunique() == 500

# ------------------------------------------------------------
# 2. Status by patient split
# ------------------------------------------------------------

print("\nSTATUS BY SPLIT")
print("-" * 60)

print(
    pd.crosstab(
        status_df["split"],
        status_df["status"]
    )
)

# ------------------------------------------------------------
# 3. Status by original dataset source
# ------------------------------------------------------------

print("\nSTATUS BY SOURCE")
print("-" * 60)

print(
    pd.crosstab(
        status_df["source"],
        status_df["status"]
    )
)

# ------------------------------------------------------------
# 4. Completed patients
# ------------------------------------------------------------

completed = status_df[
    status_df["status"] == "completed"
].copy()

completed_count = len(completed)

print("\nCOMPLETED PATIENTS:", completed_count)
print("REMAINING PATIENTS:", 500 - completed_count)

# ------------------------------------------------------------
# 5. Storage usage
# ------------------------------------------------------------

if completed_count:

    ct_sizes = pd.to_numeric(
        completed["ct_size_mb"],
        errors="coerce"
    )

    label_sizes = pd.to_numeric(
        completed["label_size_mb"],
        errors="coerce"
    )

    print("\nRECORDED STORAGE USAGE")
    print("-" * 60)

    print(
        "CT size (GiB):",
        round(ct_sizes.sum() / 1024, 3)
    )

    print(
        "Label size (GiB):",
        round(label_sizes.sum() / 1024, 3)
    )

# ------------------------------------------------------------
# 6. File inventory
# ------------------------------------------------------------

image_dir = WORK_ROOT / "images"
label_dir = WORK_ROOT / "labels"

image_files = sorted(
    image_dir.glob("*-image.npy")
)

label_files = sorted(
    label_dir.glob("*-label.npy")
)

image_ids = {
    p.name.removesuffix("-image.npy")
    for p in image_files
}

label_ids = {
    p.name.removesuffix("-label.npy")
    for p in label_files
}

complete_ids = image_ids & label_ids

print("\nOUTPUT INVENTORY")
print("-" * 60)

print("CT files:", len(image_files))
print("Label files:", len(label_files))
print("Complete patient pairs:", len(complete_ids))

# Verify consistency
assert image_ids == label_ids, (
    "Missing CT images or fracture masks detected."
)

assert set(completed["public_id"]) == complete_ids, (
    "Processing status differs from saved patient files."
)

# ------------------------------------------------------------
# 7. Create file inventory CSV
# ------------------------------------------------------------

inventory = []

for path in sorted(WORK_ROOT.rglob("*")):

    if path.is_file():

        inventory.append({
            "file": str(path.relative_to(WORK_ROOT)),
            "size_mb": round(
                path.stat().st_size / (1024**2), 3
            )
        })

inventory_df = pd.DataFrame(inventory)

inventory_path = (
    WORK_ROOT / "metadata/output_inventory.csv"
)

inventory_df.to_csv(
    inventory_path,
    index=False
)

print("\nInventory saved:", inventory_path)

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 80)

print("Total cohort:", len(status_df))
print("Completed:", completed_count)
print("Pending:", (status_df["status"] == "pending").sum())
print("Failed:", (status_df["status"] == "failed").sum())
print("Processing:", (status_df["status"] == "processing").sum())

print(
    "Completion percentage:",
    round(100 * completed_count / 500, 2), "%"
)

print("Output root:", WORK_ROOT)
print("Status file:", STATUS_PATH)

print("\nFINAL PROCESSING REPORT: PASS")
print("=" * 80)


NOTEBOOK 02 — PERSISTENT PROCESSING REPORT

PROCESSING STATUS
------------------------------------------------------------
status
pending      451
completed     49
Name: count, dtype: int64

STATUS BY SPLIT
------------------------------------------------------------
status      completed  pending
split                         
test               13       62
train              36      314
validation          0       75

STATUS BY SOURCE
------------------------------------------------------------
status       completed  pending
source                         
train_part1         49      251
train_part2          0      120
validation           0       80

COMPLETED PATIENTS: 49
REMAINING PATIENTS: 451

RECORDED STORAGE USAGE
------------------------------------------------------------
CT size (GiB): 12.001
Label size (GiB): 6.001

OUTPUT INVENTORY
------------------------------------------------------------
CT files: 49
Label files: 49
Complete patient pairs: 49

Inventory saved: /kaggl

In [90]:
#Prepare the output for publishing
from pathlib import Path
import json
import pandas as pd

OUTPUT_ROOT = Path("/kaggle/working/ribfrac_preprocessed_final")
META_ROOT = OUTPUT_ROOT / "metadata"

required_paths = [
    OUTPUT_ROOT / "images",
    OUTPUT_ROOT / "labels",
    META_ROOT / "processing_status.csv",
    META_ROOT / "patient_split_500.csv",
]

for path in required_paths:
    print(f"{'OK' if path.exists() else 'MISSING'}: {path}")

assert all(p.exists() for p in required_paths), (
    "Some required output files or directories are missing."
)

status = pd.read_csv(META_ROOT / "processing_status.csv")

print("\nPatient status:")
print(status["status"].value_counts(dropna=False))

print("\nStatus by split:")
print(pd.crosstab(status["split"], status["status"], dropna=False))

print("\nFiles ready to persist:")
print("CT arrays:", len(list((OUTPUT_ROOT / "images").glob("*-image.npy"))))
print("Label arrays:", len(list((OUTPUT_ROOT / "labels").glob("*-label.npy"))))


OK: /kaggle/working/ribfrac_preprocessed_final/images
OK: /kaggle/working/ribfrac_preprocessed_final/labels
OK: /kaggle/working/ribfrac_preprocessed_final/metadata/processing_status.csv
OK: /kaggle/working/ribfrac_preprocessed_final/metadata/patient_split_500.csv

Patient status:
status
pending       459
completed      28
failed         12
processing      1
Name: count, dtype: int64

Status by split:
status      completed  failed  pending  processing
split                                             
test                2      12       60           1
train              26       0      324           0
validation          0       0       75           0

Files ready to persist:
CT arrays: 28
Label arrays: 28


## Notebook 02 completion criteria

Notebook 02 is complete when all 500 patients have `completed` status and every completed patient has both a CT and label `.npy` file.

The persistent dataset is the input to Notebook 03. Notebook 03 should **not** contact Zenodo or recreate the patient split.

If the Kaggle working disk is insufficient for the complete 500-patient archive, process the registry in durable batches and preserve `patient_source_registry_500.csv`, `patient_split_500.csv`, `processing_status.csv`, and `preprocessing_config.json` as the audit trail.


In [86]:

OUTPUT_ROOT = Path("/kaggle/working/ribfrac_preprocessed_final")
META_ROOT = OUTPUT_ROOT / "metadata"

status_path = META_ROOT / "processing_status.csv"
split_path = META_ROOT / "patient_split_500.csv"

assert status_path.exists(), f"Missing file: {status_path}"
assert split_path.exists(), f"Missing file: {split_path}"

status = pd.read_csv(status_path)
splits = pd.read_csv(split_path)

print("PREPROCESSING STATUS")
print(status["status"].value_counts(dropna=False))

print("\nSTATUS BY SPLIT")
print(pd.crosstab(status["split"], status["status"], dropna=False))

print("\nEXPECTED PATIENT SPLITS")
print(splits["split"].value_counts())

print("\nMISSING CT OR LABEL FILES AMONG COMPLETED PATIENTS")
completed = status[status["status"] == "completed"]

missing = []
for _, row in completed.iterrows():
    ct_path = OUTPUT_ROOT / "images" / f"{row['public_id']}-image.npy"
    label_path = OUTPUT_ROOT / "labels" / f"{row['public_id']}-label.npy"

    if not ct_path.exists() or not label_path.exists():
        missing.append(row["public_id"])

print("Missing file pairs:", len(missing))
print("Examples:", missing[:10])

print("\nFAILED PATIENTS")
failed = status[status["status"] == "failed"]
print(failed[["public_id", "split", "error"]].head(15).to_string(index=False))

print("\nPENDING PATIENTS:", int((status["status"] == "pending").sum()))


PREPROCESSING STATUS
status
pending       459
completed      28
failed         12
processing      1
Name: count, dtype: int64

STATUS BY SPLIT
status      completed  failed  pending  processing
split                                             
test                2      12       60           1
train              26       0      324           0
validation          0       0       75           0

EXPECTED PATIENT SPLITS
split
train         350
validation     75
test           75
Name: count, dtype: int64

MISSING CT OR LABEL FILES AMONG COMPLETED PATIENTS
Missing file pairs: 0
Examples: []

FAILED PATIENTS
 public_id split                                                error
RibFrac114  test ValueError('too many values to unpack (expected 3)')
 RibFrac12  test ValueError('too many values to unpack (expected 3)')
RibFrac120  test ValueError('too many values to unpack (expected 3)')
RibFrac121  test ValueError('too many values to unpack (expected 3)')
RibFrac129  test ValueError('too many

In [87]:



ROOT = Path("/kaggle/working/ribfrac_preprocessed_final")
META = ROOT / "metadata"
status_path = META / "processing_status.csv"

assert status_path.exists(), f"Missing: {status_path}"

status = pd.read_csv(status_path)

print("PREPROCESSING STATUS")
print(status["status"].value_counts(dropna=False))

print("\nSTATUS BY SPLIT")
print(pd.crosstab(status["split"], status["status"], dropna=False))

completed = status[status["status"] == "completed"]
missing = []

for _, row in completed.iterrows():
    pid = str(row["public_id"])
    image = ROOT / "images" / f"{pid}-image.npy"
    label = ROOT / "labels" / f"{pid}-label.npy"

    if not image.exists() or not label.exists():
        missing.append(pid)

print("\nCompleted patients:", len(completed))
print("Completed patients missing a CT/label file:", missing)
print("\nOutput directories:")
for name in ["images", "labels", "metadata"]:
    folder = ROOT / name
    print(name, "exists:", folder.exists(),
          "| files:", len(list(folder.iterdir())) if folder.exists() else 0)


PREPROCESSING STATUS
status
pending       459
completed      28
failed         12
processing      1
Name: count, dtype: int64

STATUS BY SPLIT
status      completed  failed  pending  processing
split                                             
test                2      12       60           1
train              26       0      324           0
validation          0       0       75           0

Completed patients: 28
Completed patients missing a CT/label file: []

Output directories:
images exists: True | files: 28
labels exists: True | files: 28
metadata exists: True | files: 5


In [3]:


kagglehub.dataset_upload(
    "waboke/ribfrac-preprocessed-final",
    "/kaggle/working/ribfrac_preprocessed_final",
    version_notes="Checkpoint of current RibFrac preprocessing progress"
)

print("Upload command completed.")

Uploading Dataset https://kaggle.com/datasets/waboke/ribfrac-preprocessed-final ...


HTTPError: 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion

In [57]:
for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\n{'=' * 55}")
    print("Checking:", base)

    if not base.exists():
        print("Directory does not exist.")
        continue

    for item in sorted(base.iterdir()):
        print(f"\n{item.name}/" if item.is_dir() else item.name)

        if item.is_dir():
            for name in ["images", "labels", "metadata", "cache"]:
                sub = item / name
                if sub.is_dir():
                    print(f"  {name}: {len(list(sub.iterdir()))} entries")



Checking: /kaggle/working

.virtual_documents/

ribfrac_preprocessed_final/
  images: 0 entries
  labels: 0 entries
  metadata: 5 entries
  cache: 3 entries

Checking: /kaggle/input


In [58]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/ribfrac_preprocessed_final")

for folder in ["metadata", "cache"]:
    path = ROOT / folder
    print(f"\n{'=' * 50}\n{folder.upper()}")

    if not path.exists():
        print("Directory missing")
        continue

    for f in sorted(path.rglob("*")):
        if f.is_file():
            print(f"{f.relative_to(ROOT)} | {f.stat().st_size:,} bytes")

status_file = ROOT / "metadata" / "processing_status.csv"

if status_file.exists():
    print("\n=== PROCESSING STATUS ===")
    status = pd.read_csv(status_file)
    print("Columns:", status.columns.tolist())

    if "status" in status.columns:
        print(status["status"].value_counts(dropna=False))

    if "split" in status.columns and "status" in status.columns:
        print("\nBy split:")
        print(pd.crosstab(status["split"], status["status"]))

    print("\nFirst 10 records:")
    print(status.head(10).to_string(index=False))



METADATA
metadata/combined_ribfrac_metadata.csv | 134,956 bytes
metadata/patient_source_registry_500.csv | 14,635 bytes
metadata/patient_split_500.csv | 8,708 bytes
metadata/preprocessing_config_500.json | 369 bytes
metadata/processing_status.csv | 26,814 bytes

CACHE
cache/ribfrac-train-info-1.csv | 48,583 bytes
cache/ribfrac-train-info-2.csv | 19,816 bytes
cache/ribfrac-val-info.csv | 8,061 bytes

=== PROCESSING STATUS ===
Columns: ['public_id', 'source', 'split', 'status', 'ct_path', 'label_path', 'ct_shape', 'label_shape', 'ct_dtype', 'label_dtype', 'original_spacing', 'target_spacing', 'ct_min', 'ct_max', 'label_ids', 'ct_size_mb', 'label_size_mb', 'started_at', 'completed_at', 'error']
status
pending    500
Name: count, dtype: int64

By split:
status      pending
split              
test             75
train           350
validation       75

First 10 records:
 public_id      source      split  status  ct_path  label_path  ct_shape  label_shape  ct_dtype  label_dtype  original_s

In [5]:
input_root = Path("/kaggle/input")

print("Entries under /kaggle/input:")
for item in sorted(input_root.iterdir()):
    print(item, "(directory)" if item.is_dir() else "(file)")

print("\nSearching for persisted CT arrays...")
for path in input_root.rglob("*-image.npy"):
    print(path)


Entries under /kaggle/input:
/kaggle/input/datasets (directory)

Searching for persisted CT arrays...
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac156-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac109-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac140-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac107-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac149-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac129-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac15-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac120-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac12-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images/RibFrac114-image.npy
/kaggle/input/datasets/waboke/ribfrac-preprocessed

In [4]:
INPUT_ROOT = Path("/kaggle/input/datasets/waboke")

found = False

for root in INPUT_ROOT.rglob("ribfrac-preprocessed-final"):
    if not root.is_dir():
        continue

    images_dir = root / "images"
    labels_dir = root / "labels"

    if images_dir.is_dir() and labels_dir.is_dir():
        found = True
        ct_files = list(images_dir.glob("*-image.npy"))
        label_files = list(labels_dir.glob("*-label.npy"))

        print("Dataset root:", root)
        print("CT arrays:", len(ct_files))
        print("Label arrays:", len(label_files))
        print("Example CT files:")
        for f in sorted(ct_files)[:10]:
            print(" ", f.name)

if not found:
    print("Dataset folder not found under /kaggle/input.")
    print("Please check Add Input and confirm the dataset is attached.")


Dataset root: /kaggle/input/datasets/waboke/ribfrac-preprocessed-final
CT arrays: 13
Label arrays: 13
Example CT files:
  RibFrac107-image.npy
  RibFrac109-image.npy
  RibFrac114-image.npy
  RibFrac12-image.npy
  RibFrac120-image.npy
  RibFrac121-image.npy
  RibFrac129-image.npy
  RibFrac134-image.npy
  RibFrac140-image.npy
  RibFrac149-image.npy


In [6]:

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("=== ATTACHED DATASETS ===")

for root in INPUT_ROOT.iterdir():
    if not root.is_dir():
        continue

    ct_files = list(root.rglob("*-image.npy"))
    label_files = list(root.rglob("*-label.npy"))

    if not ct_files and not label_files:
        continue

    ct_ids = {p.name.replace("-image.npy", "") for p in ct_files}
    label_ids = {p.name.replace("-label.npy", "") for p in label_files}

    print("\nDataset root:", root)
    print("CT arrays:", len(ct_files))
    print("Label arrays:", len(label_files))
    print("Complete patient pairs:", len(ct_ids & label_ids))
    print("CT files without labels:", len(ct_ids - label_ids))
    print("Labels without CT files:", len(label_ids - ct_ids))

    print("\nFirst 10 complete patients:")
    print(sorted(ct_ids & label_ids)[:10])


=== ATTACHED DATASETS ===

Dataset root: /kaggle/input/datasets
CT arrays: 13
Label arrays: 13
Complete patient pairs: 13
CT files without labels: 0
Labels without CT files: 0

First 10 complete patients:
['RibFrac107', 'RibFrac109', 'RibFrac114', 'RibFrac12', 'RibFrac120', 'RibFrac121', 'RibFrac129', 'RibFrac134', 'RibFrac140', 'RibFrac149']


In [8]:

from pathlib import Path
import pandas as pd

# Locate the original persisted patient split
split_files = list(
    Path("/kaggle/input").rglob("patient_split_500.csv")
)

print("Found split files:")
for f in split_files:
    print(f)

if len(split_files) != 1:
    raise RuntimeError(
        f"Expected one split file, found {len(split_files)}. "
        "Check which dataset/version is attached."
    )

SPLIT_FILE = split_files[0]
splits = pd.read_csv(SPLIT_FILE)

print("\nPatient split distribution:")
print(splits["split"].value_counts())

print("\nTotal patients:", len(splits))


Found split files:
/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/metadata/patient_split_500.csv

Patient split distribution:
split
train         350
validation     75
test           75
Name: count, dtype: int64

Total patients: 500


In [9]:

from pathlib import Path
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")

ct_ids = {
    p.name.replace("-image.npy", "")
    for p in INPUT_ROOT.rglob("*-image.npy")
}

label_ids = {
    p.name.replace("-label.npy", "")
    for p in INPUT_ROOT.rglob("*-label.npy")
}

complete_ids = sorted(ct_ids & label_ids)

recovered = pd.DataFrame({
    "public_id": complete_ids
})

recovered = recovered.merge(
    splits[["public_id", "split"]],
    on="public_id",
    how="left",
    validate="one_to_one"
)

print("=== RECOVERED PATIENTS ===")
print(recovered.to_string(index=False))

print("\n=== PATIENTS BY SPLIT ===")
print(recovered["split"].value_counts(dropna=False))


=== RECOVERED PATIENTS ===
 public_id split
RibFrac107  test
RibFrac109  test
RibFrac114  test
 RibFrac12  test
RibFrac120  test
RibFrac121  test
RibFrac129  test
RibFrac134  test
RibFrac140  test
RibFrac149  test
 RibFrac15  test
RibFrac156  test
RibFrac167  test

=== PATIENTS BY SPLIT ===
split
test    13
Name: count, dtype: int64
